# K-EmoPhone Stress Prediction — CS565 IoT Data Science

| Field | Value |
|-------|-------|
| **Student** | Nadia Arvi · 20264019 |
| **Course** | CS565 IoT Data Science, KAIST |
| **Date** | June 2026 |
| **Dataset** | K-EmoPhone (Cho et al., 2022)|
| **Task** | Binary stress classification from smartphone sensor and ESM features |

---

## Abstract

This notebook implements a hierarchical evaluation of stress prediction models on the K-EmoPhone dataset:

- **Model A & B**: XGBoost baselines using sensor-only and full-feature sets
- **Model C**: Temporal LSTM encoding multi-period physiological and self-report sequences
- **Model D1**: User-conditioned LSTM with a learned per-user embedding warm-started from Personal Information Features (PIF)
- **Model D2**: Frozen LSTM backbone + lightweight per-user adapter

Two evaluation scenarios test complementary personalization hypotheses:
1. **Scenario 1 (Cold-start)**: Can stable personal traits generalize to unseen users?
2. **Scenario 2 (History-based)**: Does labeled user history improve beyond trait-informed initialization?

All preprocessing is done strictly inside cross-validation folds to prevent data leakage. Checkpoints and results are serialized to disk so the notebook can resume after a Colab disconnect.

---

## References

- Cho, M., et al. (2022). K-EmoPhone: A Mobile and Wearable Dataset with In-Situ Emotion, Stress, and Attention Labels. *Scientific Data*, 9, 351. https://doi.org/10.1038/s41597-022-01562-z
- Rule, A., et al. (2019). Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks. *PLoS Computational Biology*, 15(7), e1007007.
- Bergstra, J., et al. (2013). Making a Science of Model Search: Hyperparameter Optimization in Hundreds of Dimensions for Vision Architectures. *ICML*.
- Chawla, N. V., et al. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *JMLR*, 3, 321–357.
- Lemaître, G., et al. (2017). imbalanced-learn: A Python Toolbox to Tackle the Curse of Imbalanced Datasets in Machine Learning. *JMLR*, 18(17), 1–5.
- Paszke, A., et al. (2019). PyTorch: An Imperative Style, High-Performance Deep Learning Library. *NeurIPS*.

---

*Borrowed code: `EvXGBClassifier` and `perform_cross_validation` are adapted from the class-provided baseline notebook (original.ipynb) to ensure comparability of XGBoost results.*

## 0. Configuration

**All constants are managed in this cell.**
Centralizing configuration here makes the notebook reproducible and easy to adapt:
- Change `SEED` to test sensitivity to random initialization
- Flip `RUN_*` toggles to skip expensive sections during development
- Adjust `MAX_EPOCHS` / `PATIENCE` to trade off training time vs. convergence

Constants fall into five groups: paths, reproducibility, cross-validation, LSTM architecture, and run toggles.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# SECTION 0 — CONFIGURATION
# All experiment constants live here.  Nothing else in this notebook contains
# hard-coded values: every hyperparameter, path, and toggle is defined below.
# ─────────────────────────────────────────────────────────────────────────────
from pathlib import Path

# ── Paths ──────────────────────────────────────────────────────────────────────
# ROOT_DIR is the only path you need to change when moving the project.
# DATA_PATH and OUTPUT_DIR are derived from it automatically.
ROOT_DIR   = Path("/content/drive/MyDrive/KAIST (Graduate)/CS565 IoT Data Science/Mini Project/Models and Codes")
DATA_PATH  = str(ROOT_DIR / "features_stress_fixed_K-EmoPhone.pkl")
OUTPUT_DIR = ROOT_DIR / "Output"

# ── Output sub-directories ────────────────────────────────────────────────────
CKPT_DIR    = OUTPUT_DIR / "checkpoints"   # model weights (.pt) + prep masks (.npz)
RESULTS_DIR = OUTPUT_DIR / "results"       # per-model JSON results
HPO_DIR     = OUTPUT_DIR / "hpo"           # HPO trial pickles + results_hpo_*.json
CACHE_DIR   = OUTPUT_DIR / "cache"         # sequence / time-split cache files
FIG_DIR     = OUTPUT_DIR / "figures"       # saved plots (.png / .pdf)
for _d in [OUTPUT_DIR, CKPT_DIR, RESULTS_DIR, HPO_DIR, CACHE_DIR, FIG_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42

# ── Cross-validation ───────────────────────────────────────────────────────────
N_SPLITS  = 5
VAL_SIZE  = 0.15   # fraction of train fold used for validation / early stopping

# ── LSTM hyperparameters ───────────────────────────────────────────────────────
SEQ_STEPS   = 13
EMB_DIM     = 32
HIDDEN_SIZE = 256
N_LAYERS    = 2
DROPOUT     = 0.3
MLP_DIM     = 128
BATCH_SIZE  = 32
LR          = 1e-3
MAX_EPOCHS  = 50
PATIENCE    = 10
LR_PATIENCE = 2
LR_FACTOR   = 0.5

# ── Feature selection ──────────────────────────────────────────────────────────
LASSO_THRESHOLD = 'mean'

# ── Run toggles ────────────────────────────────────────────────────────────────
RUN_A   = True
RUN_B   = True
RUN_C   = True
RUN_D1  = True
RUN_D2  = True   # requires RUN_C = True in a prior session
RUN_HPO = True   # enable only after C/D1/D2 are complete

# ── Feature-selection comparison (Section 8 / Section 12) ─────────────────────
RUN_FS_COMPARISON = True   # sweep FS strategies for XGBoost before main A/B run

# ── Safe defaults for variables computed during training ─────────────────────
# Prevents NameError when jumping directly to visualization/results sections
# before running training cells. Overwritten by their respective sections.
best_pif      = None   # best PIF variant (set in Section 10)
a_results     = None   # Model A XGBoost results (set in Section 8)
b_results     = None   # Model B XGBoost results (set in Section 8)
c_s1_results  = None   # Model C Scenario-1 results (set in Section 9)
c_s2_results  = None   # Model C Scenario-2 results (set in Section 9)
d1_s1_results = {}     # D1 Scenario-1 results by PIF variant (set in Section 10)
d1_s2_results = None   # D1 Scenario-2 results (set in Section 10)
d2_s1_results = None   # D2 Scenario-1 results (set in Section 11)
d2_s2_results = None   # D2 Scenario-2 results (set in Section 11)

# ── Visualization constants ───────────────────────────────────────────────────
METRICS  = ['auc', 'f1', 'bal_acc', 'recall']
MLABELS  = ['AUC', 'F1', 'Bal. Acc', 'Recall']
SAVE_DPI = 300

## 1. Setup & Imports

Install dependencies, verify GPU availability, and import all third-party libraries.

**All imports are consolidated here** so any section can be re-run independently after a Colab reconnect without hitting `ImportError`.

### Key dependencies

| Library | Version (tested) | Purpose |
|---------|-----------------|---------|
| `torch` | ≥ 2.0 | LSTM models, GPU training |
| `scikit-learn` | ≥ 1.3 | Preprocessing, cross-validation, LASSO |
| `imbalanced-learn` | ≥ 0.11 | SMOTENC oversampling |
| `xgboost` | ≥ 1.7 | XGBoost baselines |
| `hyperopt` | ≥ 0.2.7 | TPE hyperparameter search |
| `pandas` / `numpy` | ≥ 2.0 / ≥ 1.24 | Data manipulation |
| `matplotlib` | ≥ 3.7 | Result visualization |

> **Reproducibility note**: `torch.backends.cudnn.deterministic = True` is set below. Combined with `SEED` seeding in Section 0, this ensures bit-identical results across runs on the same GPU/CPU configuration.

In [65]:
# ── Install missing packages (Colab only) ─────────────────────────────────────
# Pin to known-good versions to ensure reproducibility across sessions.
# If running locally, install via: pip install -r requirements.txt
!pip install -q "imbalanced-learn>=0.11" "xgboost>=1.7" "hyperopt>=0.2.7" "tqdm>=4.65"

# ── Verify GPU ────────────────────────────────────────────────────────────────
# Colab T4 or better required for training within session time limits.
!nvidia-smi

Sun Jun 14 16:58:40 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             35W /   70W |     265MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [66]:
import json, os, pickle, random, sys, time, traceback, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from IPython.display import display
from imblearn.over_sampling import SMOTE, SMOTENC
from sklearn.base import BaseEstimator, clone
from sklearn.feature_selection import SelectFromModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (balanced_accuracy_score, f1_score,
                              recall_score, roc_auc_score)
from sklearn.model_selection import (GroupShuffleSplit, StratifiedGroupKFold,
                                      train_test_split)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# ── Visualization (imported here; not deferred to Section 14) ─────────────────
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — compatible with Colab and headless envs
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


Device: cuda


## 2. Data Loading

Mount Google Drive, load the pre-extracted feature pickle, sort by user and time, and encode user IDs to contiguous integers for the embedding table.

### Dataset: K-EmoPhone
- **Participants**: 79 university students (South Korea)
- **Duration**: 7 days of continuous ambulatory monitoring
- **Sensors**: smartphone accelerometer, gyroscope, GPS, app usage, call/SMS logs
- **Self-reports (ESM)**: stress, valence, arousal, attention — prompted ~3× daily
- **Labels**: binary stress (`y = 1` = stressed, `y = 0` = not stressed) from ESM responses
- **Features**: pre-extracted time-window aggregates across 6 daily periods (Dawn → Night)

After loading, the DataFrame is sorted by `['user_id', 'datetime']` so that chronological ordering is preserved for Scenario 2 (time-split evaluation).

> **Citation**: Cho, M., et al. (2022). K-EmoPhone. *Scientific Data*, 9, 351.

In [67]:
# ── Mount Google Drive ────────────────────────────────────────────────────────
in_colab = 'google.colab' in sys.modules or Path('/content').exists()
if in_colab:
    from google.colab import drive
    drive.mount('/content/drive')

# ── Locate the feature pickle ─────────────────────────────────────────────────
# DATA_PATH (from Section 0) is checked first; the remaining entries are
# fallbacks in case the file was placed elsewhere on Drive.
FEATURE_FILE = 'features_stress_fixed_K-EmoPhone.pkl'

_candidate_paths = [
    Path(DATA_PATH),              # primary — derived from ROOT_DIR in Section 0
    ROOT_DIR / FEATURE_FILE,      # same folder as Output, without subfolder
    Path('/content') / FEATURE_FILE,
    Path('/content/data') / FEATURE_FILE,
]

_data_path = next((p for p in _candidate_paths if p.exists()), None)

if _data_path is None:
    searched = '\n'.join(f'  {p}' for p in _candidate_paths)
    raise FileNotFoundError(
        f'Cannot find {FEATURE_FILE}.\n'
        f'Searched:\n{searched}\n\n'
        f'Fix: update ROOT_DIR in Section 0, or upload the file to ROOT_DIR.'
    )

print(f'Found data at: {_data_path}')

with open(_data_path, 'rb') as f:
    X_loaded, y_loaded, groups_loaded, _, datetimes_loaded = pickle.load(f)

# ── Merge, sort, encode ───────────────────────────────────────────────────────
df_labels = pd.DataFrame({
    'user_id'  : groups_loaded,
    'datetime' : pd.to_datetime(datetimes_loaded),
    'label'    : np.asarray(y_loaded, dtype=np.float32),
})
df_merged = pd.merge(df_labels, X_loaded, left_index=True, right_index=True)
df_merged = df_merged.sort_values(['user_id', 'datetime']).reset_index(drop=True)

groups    = df_merged['user_id'].to_numpy()
y         = df_merged['label'].to_numpy(dtype=np.float32)
X_raw     = df_merged.drop(columns=['user_id', 'datetime', 'label'])

user_encoder = LabelEncoder()
user_ids     = user_encoder.fit_transform(groups).astype(np.int64)
N_USERS      = len(user_encoder.classes_)

print(f'Samples: {len(X_raw)}, Users: {N_USERS}')
print(f'Positive rate: {y.mean():.3f}')
print(f'Class counts: {np.bincount(y.astype(int))}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found data at: /content/drive/MyDrive/KAIST (Graduate)/CS565 IoT Data Science/Mini Project/Models and Codes/features_stress_fixed_K-EmoPhone.pkl
Samples: 2619, Users: 47
Positive rate: 0.350
Class counts: [1702  917]


### Fast Resume — Skip Re-training After a Colab Disconnect

After a Colab session terminates, GPU memory is lost but Google Drive files persist.

**Recovery procedure**: Run **Sections 0 → 1 → 2 → this cell**, then jump to any downstream section.

This cell reloads all previously saved JSON results from `OUTPUT_DIR`. If a result file does not exist (the model hasn't been trained yet), the corresponding variable is set to `None` and will be populated when that section runs.

Variables reloaded:
- `a_results`, `b_results` — XGBoost baseline metrics
- `c_s1_results`, `c_s2_results` — Temporal LSTM results
- `d1_s1_results` — D1 PIF ablation results (all 6 variants)
- `d1_s2_results` — D1 Scenario 2 (best PIF only)
- `d2_s1_results`, `d2_s2_results` — D2 Frozen LSTM results
- `best_pif` — recomputed from loaded D1 results, no training needed

In [68]:
def _load(path):
    p = Path(path)
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None

# ── Reload model results (JSON) ───────────────────────────────────────────────
a_results    = _load(RESULTS_DIR / 'results_model_a.json')
b_results    = _load(RESULTS_DIR / 'results_model_b.json')
c_s1_results = _load(RESULTS_DIR / 'results_model_c_s1.json')
c_s2_results = _load(RESULTS_DIR / 'results_model_c_s2.json')

PIF_CONFIGS = {
    'random'      : None,
    'demographics': None,
    'personality' : None,
    'stress_q'    : None,
    'psych'       : None,
    'all_pif'     : None,
}

d1_s1_results = {v: _load(RESULTS_DIR / f'results_model_d1_s1_{v}.json') for v in PIF_CONFIGS}
d2_s1_results = _load(RESULTS_DIR / 'results_model_d2_s1.json')
d2_s2_results = _load(RESULTS_DIR / 'results_model_d2_s2.json')
d1_s2_results = None
best_pif      = None

_d1_loaded = {k: v for k, v in d1_s1_results.items() if v is not None}
if _d1_loaded:
    best_pif = max(_d1_loaded, key=lambda k: np.mean(
        [r['auc'] for r in _d1_loaded[k]['fold_results']]))
    d1_s2_results = _load(RESULTS_DIR / f'results_model_d1_s2_{best_pif}.json')
    print(f'[resume] best_pif = {best_pif}')

# ── Reload cached numpy arrays (sequences + time-split indices) ────────────────
_CACHE_SEQ    = CACHE_DIR / 'cache_X_seq_full.npy'
_CACHE_CURR   = CACHE_DIR / 'cache_X_curr_full.npy'
_CACHE_META   = CACHE_DIR / 'cache_seq_meta.pkl'
_CACHE_SPLITS = CACHE_DIR / 'cache_time_splits.pkl'

if _CACHE_SEQ.exists() and _CACHE_CURR.exists() and _CACHE_META.exists():
    X_seq_full  = np.load(_CACHE_SEQ)
    X_curr_full = np.load(_CACHE_CURR)
    with open(_CACHE_META, 'rb') as _f:
        _step_groups, _curr_cols, seq_cat_mask, curr_cat_mask = pickle.load(_f)
    print(f'[resume] sequences: X_seq_full={X_seq_full.shape}, X_curr_full={X_curr_full.shape}')
else:
    X_seq_full = X_curr_full = _step_groups = _curr_cols = seq_cat_mask = curr_cat_mask = None
    print('[resume] sequences not cached yet — run Section 9 first')

if _CACHE_SPLITS.exists():
    with open(_CACHE_SPLITS, 'rb') as _f:
        TIME_TRAIN_IDX, TIME_TEST_IDX = pickle.load(_f)
    print(f'[resume] time-split: train={len(TIME_TRAIN_IDX)}, test={len(TIME_TEST_IDX)}')
else:
    TIME_TRAIN_IDX = TIME_TEST_IDX = None
    print('[resume] time-split indices not cached yet — run Section 9 first')

# ── HPO results (loaded if available; None if HPO hasn't run yet) ────────────
b_hpo_results  = _load(HPO_DIR / 'results_hpo_b.json')
best_hpo_d1    = None
best_hpo_d2    = None

# ── Status summary ─────────────────────────────────────────────────────────────
for name, res in [('A', a_results), ('B', b_results), ('B+HPO', b_hpo_results),
                   ('C·S1', c_s1_results), ('C·S2', c_s2_results),
                   ('D2·S1', d2_s1_results), ('D2·S2', d2_s2_results)]:
    print(f'  {name}: {"✓" if res else "✗"}')
for v, res in d1_s1_results.items():
    print(f'  D1·S1·{v}: {"✓" if res else "✗"}')


[resume] best_pif = psych
[resume] sequences: X_seq_full=(2619, 13, 417), X_curr_full=(2619, 85)
[resume] time-split: train=1814, test=805
  A: ✓
  B: ✓
  C·S1: ✓
  C·S2: ✓
  D2·S1: ✓
  D2·S2: ✓
  D1·S1·random: ✓
  D1·S1·demographics: ✓
  D1·S1·personality: ✓
  D1·S1·stress_q: ✓
  D1·S1·psych: ✓
  D1·S1·all_pif: ✓


## 3. Feature Engineering

Split raw features into semantically meaningful groups that serve different roles in the pipeline.

### Feature group taxonomy

| Group | Role | Models |
|-------|------|--------|
| `feat_baseline` | Sensor features only (current + past periods) | A (XGBoost) |
| `feat_temporal` | Sensor + ESM self-reports across 6 daily periods | C, D1, D2 (LSTM input) |
| `feat_pif_all` | Static personal traits (personality, demographics, questionnaires) | D1 embedding init |
| `feat_all` | Everything concatenated | B (XGBoost) |

### Why PIF must NOT appear in `feat_temporal`

PIF features (e.g., Big Five personality scores, age, PSS questionnaire responses) are collected **once at enrollment** — they are constant for every row belonging to the same user. Including them in LSTM sequences would repeat the same value at every time step, adding noise without information. Instead, PIF is used exclusively to **initialize** the per-user embedding in D1, projecting static trait knowledge into the latent user representation.

### Period ordering for sequence construction

The K-EmoPhone feature names encode the time window: `Dawn`, `Morning`, `Afternoon`, `LateAfternoon`, `Evening`, `Night`. This ordering forms the **13-step temporal sequence** fed to the LSTM encoder (6 periods × today + 6 periods × yesterday + 1 current step).

In [69]:
X = X_raw  # alias for readability

# ── Sensor / ESM splits by time window ────────────────────────────────────────
# Use column-name filtering (not boolean lists) for pandas >= 2.0 compatibility
feat_current_sensor       = X[[c for c in X.columns if '#VAL' in c and 'ESM' not in c]]
feat_current_ESM          = X[[c for c in X.columns if 'ESM#LastLabel' in c]]
feat_ImmediatePast_raw    = X[[c for c in X.columns if 'ImmediatePast_15' in c]]
feat_ImmediatePast_sensor = feat_ImmediatePast_raw[[c for c in feat_ImmediatePast_raw.columns if 'ESM' not in c]]
feat_ImmediatePast_ESM    = feat_ImmediatePast_raw[[c for c in feat_ImmediatePast_raw.columns if 'ESM' in c]]
feat_today_raw            = X[[c for c in X.columns if 'Today' in c]]
feat_today_sensor         = feat_today_raw[[c for c in feat_today_raw.columns if 'ESM' not in c]]
feat_today_ESM            = feat_today_raw[[c for c in feat_today_raw.columns if 'ESM' in c]]
feat_yesterday_raw        = X[[c for c in X.columns if 'Yesterday' in c]]
feat_yesterday_sensor     = feat_yesterday_raw[[c for c in feat_yesterday_raw.columns if 'ESM' not in c]]
feat_yesterday_ESM        = feat_yesterday_raw[[c for c in feat_yesterday_raw.columns if 'ESM' in c]]

# ── Composed feature sets ─────────────────────────────────────────────────────
feat_baseline = pd.concat([
    feat_current_sensor, feat_ImmediatePast_sensor,
    feat_today_sensor,   feat_yesterday_sensor,
], axis=1)  # sensor only — Model A

feat_temporal = pd.concat([
    feat_baseline,
    feat_current_ESM, feat_ImmediatePast_ESM,
    feat_today_ESM,   feat_yesterday_ESM,
], axis=1)  # sensor + ESM — LSTM input

# ── PIF feature groups (static per user, collected at enrollment) ──────────────
feat_pif_all = X[[c for c in X.columns if 'PIF' in c]]

# Sub-group detection: adapt to actual column names in the dataset
feat_demographics  = X[[c for c in X.columns if 'PIF' in c and any(k in c for k in ['AGE','GEN'])]]
feat_personality   = X[[c for c in X.columns if 'PIF' in c and 'BFI' in c]]
feat_stress_q      = X[[c for c in X.columns if 'PIF' in c and 'PSS' in c]]
feat_mental_health = X[[c for c in X.columns if 'PIF' in c and any(k in c for k in ['PHQ','GHQ'])]]

# ── Combined (Model B only) ────────────────────────────────────────────────────
feat_all = pd.concat([feat_temporal, feat_pif_all], axis=1)

# ── Categorical column detection (pd.api.types for pandas version safety) ─────
def _is_bool_col(series):
    return pd.api.types.is_bool_dtype(series) or (
        series.dropna().unique().tolist() in ([0, 1], [1, 0], [0.0, 1.0], [1.0, 0.0]))

C_cat_baseline = np.asarray(sorted(c for c in feat_baseline.columns if _is_bool_col(feat_baseline[c])))
C_num_baseline = np.asarray(sorted(c for c in feat_baseline.columns if not _is_bool_col(feat_baseline[c])))
C_cat_all      = np.asarray(sorted(c for c in feat_all.columns     if _is_bool_col(feat_all[c])))
C_num_all      = np.asarray(sorted(c for c in feat_all.columns     if not _is_bool_col(feat_all[c])))

print(f'feat_baseline : {feat_baseline.shape}')
print(f'feat_temporal : {feat_temporal.shape}')
print(f'feat_pif_all  : {feat_pif_all.shape}')
print(f'feat_all      : {feat_all.shape}')
print(f'PIF sub-groups: demographics={feat_demographics.shape[1]}, '
      f'personality={feat_personality.shape[1]}, '
      f'stress_q={feat_stress_q.shape[1]}, '
      f'mental_health={feat_mental_health.shape[1]}')


feat_baseline : (2619, 5492)
feat_temporal : (2619, 5505)
feat_pif_all  : (2619, 11)
feat_all      : (2619, 5516)
PIF sub-groups: demographics=0, personality=5, stress_q=1, mental_health=1


## 4. Evaluation Strategy

### Two Evaluation Scenarios

This project evaluates the user-conditioned LSTM under two complementary scenarios
that test different types of personalization:

**Scenario 1 — Cold-start (trait-informed):** 5-fold StratifiedGroupKFold with
user-disjoint splits.
- Test users are completely unseen during training
- Embeddings are initialized from PIF features (personality, demographics, questionnaire data) but receive no gradient updates
- **Goal** : Measures whether stable personal traitsalone are sufficient to adapt predictions for a new user.

**Scenario 2 — History-based (known-user):** Per-user chronological time-split
(first 70% → train, last 30% → test).
- All 79 users appear in both splits
- Embeddings are trained on each user's early labeled data, then evaluated on their future stress
- **Goal** : Measures whether historical stress data improves future prediction beyond
trait-informed initialization.

### Why two scenarios?

Scenario 1 isolates the benefit of **stable personal traits** (cold-start personalization), while Scenario 2 adds **labeled historical data** (warm-start personalization). Comparing the two AUC deltas reveals whether collecting longitudinal labels per user is worth the annotation cost.

### Metrics

| Metric | Why it matters |
|--------|---------------|
| **AUC-ROC** | Primary; robust to class imbalance |
| **F1** | Balances precision and recall for stress class |
| **Balanced Accuracy** | Equal weight to each class — stress is minority |
| **Recall** | Stress detection sensitivity; missing stress is costly |

Stress labels are imbalanced (~30–40% positive rate in K-EmoPhone), so accuracy alone is misleading. AUC is the primary ranking metric.

## 5. Shared Preprocessing Helpers

**Function definitions only — no execution happens here.**

### Inside-fold preprocessing order (strictly enforced to prevent leakage)

```
1. StandardScaler  → fit on X_train only → transform X_train / X_val / X_test
2. LASSO selector  → fit on X_train_scaled → select features for all splits
3. SMOTENC         → fit_resample on X_train only → never applied to val or test
4. build_sequences → reshape flat features → (N, SEQ_STEPS, n_feat_per_step)
```

**Why this order matters**: fitting any transformer on val/test data leaks the target distribution into preprocessing, inflating all downstream metrics. The order above ensures every transformation decision is made using only information available at training time.

### Helper summary

| Function | Purpose |
|----------|---------|
| `run_or_load` | Cache-and-resume wrapper for expensive training calls |
| `is_categorical` | Detect bool/binary columns for SMOTENC |
| `build_sequences` | Reshape flat features into (N, T, F) LSTM input |
| `build_pif_matrix` | Build (n_users, pif_dim) tensor for embedding init |
| `time_split_indices` | Per-user chronological train/test split |
| `fit_lasso_seq_masks` | LASSO feature selection on sequence data |
| `oversample_with_users` | SMOTENC wrapper preserving user_id column |

In [70]:
# ── run_or_load ───────────────────────────────────────────────────────────────
def run_or_load(result_path: Path, compute_fn, force_rerun: bool = False):
    """Load results from disk if they exist, otherwise run compute_fn and save."""
    if result_path.exists() and not force_rerun:
        print(f'[resume] Loading {result_path.name}')
        with open(result_path) as f:
            return json.load(f)
    print(f'[train]  Running and saving to {result_path.name}')
    results = compute_fn()
    with open(result_path, 'w') as f:
        json.dump(results, f, indent=2)
    return results


In [71]:
# ── Column helpers ────────────────────────────────────────────────────────────
def is_categorical(series):
    """Return True for bool or binary 0/1 columns."""
    if pd.api.types.is_bool_dtype(series):
        return True
    vals = pd.Series(series.dropna().unique())
    return len(vals) <= 2 and set(vals.astype(str)).issubset({'0','1','0.0','1.0','False','True'})


def get_cat_num_arrays(df):
    """Return (C_cat, C_num) — sorted column name arrays."""
    C_cat = np.asarray(sorted(c for c in df.columns if is_categorical(df[c])))
    C_num = np.asarray(sorted(c for c in df.columns if not is_categorical(df[c])))
    return C_cat, C_num


In [72]:
# ── Temporal sequence construction ────────────────────────────────────────────
PERIODS = ['Dawn', 'Morning', 'Afternoon', 'LateAfternoon', 'Evening', 'Night']

def build_sequences(X_flat, seq_steps=SEQ_STEPS):
    """
    Reshape flat per-sample features into temporal sequences.

    Groups feat_temporal columns by period (Yesterday x6, Today x6, ImmediatePast x1)
    to form time steps. Current-window features (#VAL, ESM#LastLabel) become X_curr.

    Args:
        X_flat    : DataFrame of temporal features (feat_temporal)
        seq_steps : number of time steps (default SEQ_STEPS)
    Returns:
        X_seq      : np.ndarray (N, seq_steps, n_feat_per_step) — encoder input
        X_curr     : np.ndarray (N, curr_dim)                   — decoder input
        seq_col_groups : list[list[str]]  — column names per step (for masks)
        curr_cols  : list[str]            — columns in X_curr
        seq_cat_mask  : np.ndarray(bool, n_feat_per_step)
        curr_cat_mask : np.ndarray(bool, curr_dim)
    """
    cols = list(X_flat.columns)
    col_to_idx = {c: i for i, c in enumerate(cols)}
    values = np.nan_to_num(X_flat.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
    N = len(X_flat)

    # Assign columns to time steps
    step_col_groups = []
    for p in PERIODS:
        step_col_groups.append([c for c in cols if f'Yesterday{p}' in c])
    for p in PERIODS:
        step_col_groups.append([c for c in cols if f'Today{p}' in c])
    step_col_groups.append([c for c in cols if 'ImmediatePast_15' in c])  # step 12

    assert len(step_col_groups) == seq_steps, \
        f'Expected {seq_steps} step groups, got {len(step_col_groups)}'

    # Current features (right before the label, not part of history window)
    curr_cols = [c for c in cols if '#VAL' in c or 'ESM#LastLabel' in c]

    # Pad all steps to the maximum step feature count
    n_feat = max((len(g) for g in step_col_groups), default=1)

    X_seq = np.zeros((N, seq_steps, n_feat), dtype=np.float32)
    for step, step_cols in enumerate(step_col_groups):
        if step_cols:
            idx = [col_to_idx[c] for c in step_cols if c in col_to_idx]
            X_seq[:, step, :len(idx)] = values[:, idx]

    curr_idx = [col_to_idx[c] for c in curr_cols if c in col_to_idx]
    X_curr = values[:, curr_idx].astype(np.float32) if curr_idx else np.zeros((N, 1), dtype=np.float32)

    # Categorical masks (bool dtype columns)
    bool_set = set(X_flat.columns[X_flat.dtypes == bool].tolist())
    ref_cols = max(step_col_groups, key=len)  # reference: the longest step
    seq_cat_mask = np.zeros(n_feat, dtype=bool)
    for i, c in enumerate(ref_cols):
        if i < n_feat and c in bool_set:
            seq_cat_mask[i] = True
    curr_cat_mask = np.array([c in bool_set for c in curr_cols], dtype=bool)

    return X_seq, X_curr, step_col_groups, curr_cols, seq_cat_mask, curr_cat_mask


In [73]:
# ── PIF lookup matrix ─────────────────────────────────────────────────────────
def build_pif_matrix(df_full, user_enc, pif_cols, user_id_col='user_id'):
    """
    Build a (n_users, len(pif_cols)) float32 tensor of per-user PIF features.

    Uses groupby.first() since PIF features are static per user (enrollment data).
    Rows are ordered to match user_enc.classes_ so row i = encoded user i.

    Args:
        df_full     : full merged DataFrame with user_id column
        user_enc    : fitted LabelEncoder for user IDs
        pif_cols    : list of PIF column names (empty list → return None)
        user_id_col : name of the user ID column in df_full
    Returns:
        Tensor (n_users, pif_dim) or None if pif_cols is empty
    """
    if not pif_cols:
        return None
    pif_per_user = df_full.groupby(user_id_col)[pif_cols].first()
    ordered = pif_per_user.reindex(user_enc.classes_).fillna(0.0).values.astype(np.float32)
    return torch.tensor(ordered, dtype=torch.float32)


In [74]:
# ── Chronological time-split indices ─────────────────────────────────────────
def time_split_indices(df, train_ratio=0.7, user_id_col='user_id'):
    """
    Per-user chronological split. df must be pre-sorted by [user_id, datetime].

    Each user's first train_ratio samples go to train; the rest go to test.
    Returns integer positional indices (not labels) for use with .iloc[].

    Args:
        df          : DataFrame sorted by [user_id, datetime]
        train_ratio : fraction of each user's samples for training
        user_id_col : user identifier column name
    Returns:
        train_idx : list[int]
        test_idx  : list[int]
    """
    train_idx, test_idx = [], []
    for uid in df[user_id_col].unique():
        positions = np.where(df[user_id_col].values == uid)[0].tolist()
        n_train = max(1, int(len(positions) * train_ratio))
        train_idx.extend(positions[:n_train])
        test_idx.extend(positions[n_train:])
    return train_idx, test_idx


In [75]:
# ── LASSO mask for sequences ──────────────────────────────────────────────────
def fit_lasso_seq_masks(seq_tr, curr_tr, y_tr, threshold=None):
    """
    Fit LogReg LASSO on flattened [seq, curr] and return per-feature boolean masks.

    Averages absolute coefficients across time steps so the same feature subset
    is selected for every step — ensuring (N, S, n_sel) has a uniform last dim.

    Args:
        seq_tr  : (N, S, F) scaled training sequences
        curr_tr : (N, C) scaled current features
        y_tr    : (N,) binary labels
    Returns:
        seq_keep  : bool array (F,)
        curr_keep : bool array (C,)
    """
    N, S, F = seq_tr.shape
    flat = np.concatenate([seq_tr.reshape(N, S * F), curr_tr], axis=1)

    # ⚠️ LEAKAGE RISK: LASSO selector fitted on train only — test features must not influence selection
    selector = SelectFromModel(
        LogisticRegression(penalty='l1', solver='liblinear', C=1,
                           random_state=SEED, max_iter=4000),
        threshold=threshold if threshold is not None else LASSO_THRESHOLD,
    )
    selector.fit(flat, y_tr)

    coefs = np.abs(selector.estimator_.coef_[0])
    seq_imp  = coefs[:S * F].reshape(S, F).mean(axis=0)  # avg importance across steps
    curr_imp = coefs[S * F:]

    global_mean = np.concatenate([seq_imp, curr_imp]).mean()
    seq_keep  = seq_imp  > global_mean
    curr_keep = curr_imp > global_mean if len(curr_imp) > 0 else np.ones(curr_tr.shape[1], dtype=bool)

    # Guarantee at least a few features survive
    if not seq_keep.any():
        seq_keep[np.argsort(seq_imp)[-min(32, F):]] = True
    if not curr_keep.any() and curr_tr.shape[1] > 0:
        curr_keep[np.argsort(curr_imp)[-min(8, len(curr_imp)):]] = True

    return seq_keep, curr_keep


def scale_flat(tr, va, te, cat_mask):
    """Standard-scale numeric columns; leave boolean columns untouched."""
    num = ~cat_mask
    tr_s, va_s, te_s = tr.copy(), va.copy(), te.copy()
    if num.any():
        # ⚠️ LEAKAGE RISK: scaler fitted on train only — fitting on val/test leaks future distribution
        sc = StandardScaler().fit(tr[:, num])
        tr_s[:, num] = sc.transform(tr[:, num])
        va_s[:, num] = sc.transform(va[:, num])
        te_s[:, num] = sc.transform(te[:, num])
    return tr_s.astype(np.float32), va_s.astype(np.float32), te_s.astype(np.float32)


def preprocess_lstm_fold(seq_tr, curr_tr, seq_va, curr_va, seq_te, curr_te,
                          y_tr, seq_cat_mask, curr_cat_mask,
                          lasso_threshold=None,
                          fixed_seq_keep=None, fixed_curr_keep=None):
    """
    Full inside-fold preprocessing for LSTM models.

    Order: scale → LASSO → return masks + scaled arrays.
    SMOTENC is applied by the caller after this returns (to preserve user_ids handling).

    Returns scaled + masked arrays and a prep dict for reuse (D2 compatibility).
    """
    N, S, F = seq_tr.shape
    # Flatten per-step for scaling
    sfl_tr = seq_tr.reshape(-1, F)
    sfl_va = seq_va.reshape(-1, F)
    sfl_te = seq_te.reshape(-1, F)

    sfl_tr_s, sfl_va_s, sfl_te_s = scale_flat(sfl_tr, sfl_va, sfl_te, seq_cat_mask)
    seq_tr_s = sfl_tr_s.reshape(seq_tr.shape)
    seq_va_s = sfl_va_s.reshape(seq_va.shape)
    seq_te_s = sfl_te_s.reshape(seq_te.shape)

    curr_tr_s, curr_va_s, curr_te_s = scale_flat(curr_tr, curr_va, curr_te, curr_cat_mask)

    if fixed_seq_keep is not None:
        seq_keep, curr_keep = fixed_seq_keep, fixed_curr_keep
    else:
        seq_keep, curr_keep = fit_lasso_seq_masks(seq_tr_s, curr_tr_s, y_tr, threshold=lasso_threshold)

    prep = {
        'seq_keep'         : seq_keep,
        'curr_keep'        : curr_keep,
        'seq_cat_selected' : seq_cat_mask[seq_keep],
        'curr_cat_selected': curr_cat_mask[curr_keep],
    }

    return (
        seq_tr_s[:, :, seq_keep], curr_tr_s[:, curr_keep],
        seq_va_s[:, :, seq_keep], curr_va_s[:, curr_keep],
        seq_te_s[:, :, seq_keep], curr_te_s[:, curr_keep],
        prep,
    )


def apply_prep(seq, curr, prep, seq_cat_mask, curr_cat_mask):
    """Apply a saved preprocessor to new split (used by D2 to share Model-C dims)."""
    N, S, F = seq.shape
    sfl = seq.reshape(-1, F)
    num_s = ~seq_cat_mask
    sfl_s = sfl.copy()
    if num_s.any():
        sc = StandardScaler()
        # reconstruct scaler from stored data — we refit on supplied data instead
        # (D2 must call preprocess_lstm_fold on its own splits then apply prep masks)
        pass
    return seq[:, :, prep['seq_keep']], curr[:, prep['curr_keep']]


In [76]:
# ── SMOTENC wrapper that preserves user_ids ───────────────────────────────────
def oversample_with_users(seq_tr, curr_tr, y_tr, users_tr, seq_cat_sel, curr_cat_sel):
    """
    Apply SMOTENC to flattened [seq, curr, user_id]; reshape back to sequences.

    user_id is appended as a categorical column so synthetic samples get valid IDs.

    Args:
        seq_tr, curr_tr : scaled masked arrays
        y_tr            : binary labels (float)
        users_tr        : integer user IDs
        seq_cat_sel     : bool mask (seq feature dim) post-LASSO
        curr_cat_sel    : bool mask (curr feature dim) post-LASSO
    Returns:
        seq, curr, y, users — oversampled and reshaped
    """
    N, S, F = seq_tr.shape
    flat = np.concatenate([
        seq_tr.reshape(N, S * F),
        curr_tr,
        users_tr.reshape(-1, 1).astype(np.float32),
    ], axis=1)
    cat_flags = np.concatenate([
        np.tile(seq_cat_sel, S),
        curr_cat_sel,
        [True],  # user_id is categorical
    ])
    cat_indices = np.where(cat_flags)[0].tolist()

    k = max(1, min(5, int(np.bincount(y_tr.astype(int)).min()) - 1))
    # ⚠️ LEAKAGE RISK: SMOTENC on train only — oversampling val/test inflates performance artificially
    sampler = SMOTENC(categorical_features=cat_indices, random_state=SEED, k_neighbors=k) \
        if cat_indices else SMOTE(random_state=SEED, k_neighbors=k)
    flat_res, y_res = sampler.fit_resample(flat, y_tr.astype(int))

    seq_res  = flat_res[:, :S*F].reshape(-1, S, F).astype(np.float32)
    curr_res = flat_res[:, S*F:-1].astype(np.float32)
    uid_res  = np.clip(np.rint(flat_res[:, -1]).astype(np.int64), 0, N_USERS - 1)
    return seq_res, curr_res, y_res.astype(np.float32), uid_res


def train_val_split(tv_idx, y_arr, grp_arr, fold, val_size=VAL_SIZE):
    """GroupShuffleSplit within the train-fold to get a held-out validation set."""
    for offset in range(50):
        spl = GroupShuffleSplit(n_splits=1, test_size=val_size, random_state=SEED + fold * 100 + offset)
        tr_rel, va_rel = next(spl.split(tv_idx, y_arr[tv_idx], grp_arr[tv_idx]))
        tr_idx = tv_idx[tr_rel]
        va_idx = tv_idx[va_rel]
        # ensure both train and val have at least one of each class
        if len(np.unique(y_arr[tr_idx])) > 1 and len(np.unique(y_arr[va_idx])) > 1:
            return tr_idx, va_idx
    return tv_idx[tr_rel], tv_idx[va_rel]


## 6. Model Definitions

### Model architecture overview

```
Model C — TemporalLSTM
  LSTM(input=seq_dim, hidden=HIDDEN_SIZE, layers=N_LAYERS, dropout=DROPOUT)
  → last hidden state h_n
  → MLP: Linear(HIDDEN_SIZE + curr_dim → MLP_DIM) → ReLU → Dropout → Linear(MLP_DIM → 1) → Sigmoid

Model D1 — UserConditionedLSTM  [extends Model C with personalization]
  nn.Embedding(n_users, EMB_DIM)  ← Xavier init OR warm-started from PIF
  Injection ①: concat(e_u, x_t) at every encoder timestep  → LSTM sees user at each step
  Injection ②: concat(e_u, h_n, x_curr) at MLP input      → threshold adapts per user

Model D2 — FrozenLSTMWithEmbedding  [transfer learning from Model C]
  Load TemporalLSTM weights → freeze all LSTM params
  Train only: nn.Embedding(n_users, EMB_DIM) + Adapter(HIDDEN_SIZE + EMB_DIM → 64 → 1)
```

### Design rationale

**D1 vs D2**: D1 trains the full model jointly with user conditioning — the LSTM itself learns to process information differently per user. D2 asks whether a frozen general-purpose LSTM can be personalized post-hoc using only a small adapter. D2 is a more deployment-friendly approach (the backbone is not retrained per dataset).

**PIF warm-start**: initializing D1's embedding from PIF features (via `init_from_pif`) gives the model a structured starting point rather than random noise. The PIF ablation in Section 10 identifies which trait categories carry the most signal.

In [77]:
class StressDataset(Dataset):
    """Unified dataset for all LSTM models. Accepts optional user_ids."""

    def __init__(self, seq, curr, labels, user_ids=None):
        self.seq      = torch.as_tensor(seq,    dtype=torch.float32)
        self.curr     = torch.as_tensor(curr,   dtype=torch.float32)
        self.labels   = torch.as_tensor(labels, dtype=torch.float32)
        _uids = user_ids if user_ids is not None else np.zeros(len(labels), dtype=np.int64)
        self.user_ids = torch.as_tensor(_uids,  dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.seq[idx], self.curr[idx], self.user_ids[idx], self.labels[idx]


In [78]:
class TemporalLSTM(nn.Module):
    """
    Model C: temporal LSTM encoder + MLP decoder for stress prediction.

    No user conditioning. Serves as the baseline and as the backbone for D2.

    Args:
        seq_dim     : input features per time step after LASSO selection
        curr_dim    : dimension of the current-step (decoder) input
        hidden_size : LSTM hidden dimension
        n_layers    : number of stacked LSTM layers
        dropout     : dropout probability in LSTM inter-layer and MLP
    """

    def __init__(self, seq_dim, curr_dim,
                 hidden_size=HIDDEN_SIZE, n_layers=N_LAYERS, dropout=DROPOUT):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=seq_dim, hidden_size=hidden_size,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + curr_dim, MLP_DIM),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(MLP_DIM, 1),
            nn.Sigmoid(),
        )

    def forward(self, x_seq, x_curr, user_ids=None):
        _, (h, _) = self.lstm(x_seq)
        return self.mlp(torch.cat([h[-1], x_curr], dim=1)).squeeze(1)


In [79]:
class UserConditionedLSTM(nn.Module):
    """
    Model D1: per-user embedding injected at every encoder timestep and the decoder.

    Injection ①: embedding e_u concatenated to every LSTM input step.
    Injection ②: e_u concatenated with h_n and x_curr at the MLP input.

    PIF warm-start: if pif_dim is provided, a projection layer maps PIF features
    to the embedding space; init_from_pif() copies those projections to the
    embedding table before training begins (cold-start initialization).

    Args:
        seq_dim   : LASSO-selected features per time step
        curr_dim  : LASSO-selected current features
        n_users   : total number of unique users
        emb_dim   : user embedding dimension
        pif_dim   : dimension of PIF features used for warm-start (None = random Xavier)
    """

    def __init__(self, seq_dim, curr_dim, n_users=N_USERS,
                 emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE,
                 n_layers=N_LAYERS, dropout=DROPOUT, pif_dim=None):
        super().__init__()
        self.emb_dim = emb_dim
        self.user_emb = nn.Embedding(n_users, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)

        if pif_dim is not None:
            self.pif_proj = nn.Linear(pif_dim, emb_dim)
        else:
            self.pif_proj = None

        # Injection ①: seq_dim + emb_dim per step
        self.lstm = nn.LSTM(
            input_size=seq_dim + emb_dim, hidden_size=hidden_size,
            num_layers=n_layers, batch_first=True,
            dropout=dropout if n_layers > 1 else 0.0,
        )
        # Injection ②: hidden + curr_dim + emb_dim
        self.mlp = nn.Sequential(
            nn.Linear(hidden_size + curr_dim + emb_dim, MLP_DIM),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(MLP_DIM, 1),
            nn.Sigmoid(),
        )

    def init_from_pif(self, pif_matrix):
        """
        Warm-start user embeddings from PIF features.

        Called once before training. pif_matrix: (n_users, pif_dim) on device.
        After this call, embeddings encode trait information as their prior.
        """
        assert self.pif_proj is not None, 'pif_dim=None — cannot init from PIF'
        with torch.no_grad():
            self.user_emb.weight.copy_(self.pif_proj(pif_matrix))

    def forward(self, x_seq, x_curr, user_ids):
        e = self.user_emb(user_ids)  # (B, emb_dim)
        # Injection ①: tile embedding across all time steps
        e_expanded = e.unsqueeze(1).expand(-1, x_seq.size(1), -1)
        conditioned = torch.cat([x_seq, e_expanded], dim=-1)
        _, (h, _) = self.lstm(conditioned)
        # Injection ②: concat at decoder
        return self.mlp(torch.cat([h[-1], x_curr, e], dim=1)).squeeze(1)


In [80]:
class FrozenLSTMWithEmbedding(nn.Module):
    """
    Model D2: frozen Model-C LSTM + trainable user embedding + lightweight adapter.

    Phase 1: load TemporalLSTM weights from a checkpoint.
    Phase 2: freeze all LSTM parameters; train only embedding + adapter.

    The LSTM forward pass runs under torch.no_grad() for efficiency.

    Args:
        temporal_model : a TemporalLSTM instance with loaded C weights
        n_users        : total unique users
        emb_dim        : user embedding dimension
        hidden_size    : must match temporal_model.lstm.hidden_size
    """

    def __init__(self, temporal_model, n_users=N_USERS,
                 emb_dim=EMB_DIM, hidden_size=HIDDEN_SIZE):
        super().__init__()
        self.lstm = temporal_model.lstm
        # NOTE: D2 freezes LSTM weights here — only embedding + adapter receive gradients
        for p in self.lstm.parameters():
            p.requires_grad = False

        self.user_emb = nn.Embedding(n_users, emb_dim)
        nn.init.xavier_uniform_(self.user_emb.weight)

        self.adapter = nn.Sequential(
            nn.Linear(hidden_size + emb_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x_seq, x_curr, user_ids):
        with torch.no_grad():
            _, (h, _) = self.lstm(x_seq)
        e = self.user_emb(user_ids)
        return self.adapter(torch.cat([h[-1], e], dim=1)).squeeze(1)


## 7. Training & Evaluation Helpers

**Function definitions only.**

### Cross-validation functions

| Function | Scenario | Split strategy |
|----------|----------|---------------|
| `run_group_cv` | Scenario 1 | `StratifiedGroupKFold(n_splits=N_SPLITS)` — user-disjoint folds |
| `run_time_split` | Scenario 2 | Per-user chronological 70/30 split |

### Training loop per fold

```
optimizer : Adam(lr=LR)
scheduler : ReduceLROnPlateau(mode='max', patience=LR_PATIENCE, factor=LR_FACTOR)
criterion : BCELoss()
early stop: patience=PATIENCE on val AUC — saves best checkpoint
```

### Why StratifiedGroupKFold for Scenario 1?

Stratification preserves the stress/non-stress ratio in each fold (important given class imbalance). The `groups=user_ids` argument guarantees that all samples from a user land in the same fold — preventing any user identity leakage between train and test.

### Leakage safeguards in `run_group_cv`

Every fold independently: (1) fits scaler on train split, (2) fits LASSO on scaled train, (3) applies SMOTENC on train only, (4) builds sequences from selected features. The val split is used only for early stopping; the test split is touched exactly once at final evaluation.

In [81]:
def make_loader(seq, curr, user_ids, labels, shuffle=False):
    """Wrap arrays in a DataLoader with the global BATCH_SIZE."""
    return DataLoader(
        StressDataset(seq, curr, labels, user_ids),
        batch_size=BATCH_SIZE, shuffle=shuffle,
    )


def compute_metrics(y_true, y_prob):
    """Compute AUC, F1, balanced accuracy, and recall from probabilities."""
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'auc'     : float(roc_auc_score(y_true, y_prob)),
        'f1'      : float(f1_score(y_true, y_pred, zero_division=0)),
        'bal_acc' : float(balanced_accuracy_score(y_true, y_pred)),
        'recall'  : float(recall_score(y_true, y_pred, zero_division=0)),
    }


def predict_proba(model, loader):
    """Run inference and return (y_true, y_prob) numpy arrays."""
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for seq, curr, users, labels in loader:
            prob = model(seq.to(device), curr.to(device), users.to(device))
            preds.extend(prob.cpu().numpy())
            targets.extend(labels.numpy())
    return np.array(targets), np.array(preds)


In [82]:
def train_fold_lstm(model, train_loader, val_loader, ckpt_path,
                     max_epochs=MAX_EPOCHS, patience=PATIENCE):
    """
    Training loop with early stopping on val AUC and ReduceLROnPlateau scheduling.

    Saves best checkpoint to ckpt_path. Returns (best_val_auc, history_list).

    Args:
        model        : nn.Module to train (already .to(device) by caller)
        train_loader : DataLoader for training data
        val_loader   : DataLoader for validation data (early stopping only)
        ckpt_path    : Path where best model state is saved
        max_epochs   : maximum training epochs
        patience     : early stopping patience on val AUC
    """
    model.to(device)
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer  = torch.optim.Adam(params, lr=LR)
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', patience=LR_PATIENCE, factor=LR_FACTOR)
    criterion  = nn.BCELoss()

    best_auc, best_state, wait = -np.inf, None, 0
    history = []

    for epoch in range(1, max_epochs + 1):
        torch.manual_seed(SEED + epoch)
        model.train()
        losses = []
        for seq, curr, users, labels in train_loader:
            seq, curr, users, labels = (
                seq.to(device), curr.to(device), users.to(device), labels.to(device))
            optimizer.zero_grad()
            pred = model(seq, curr, users)
            loss = criterion(pred, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(params, max_norm=5.0)
            optimizer.step()
            losses.append(loss.item())

        val_y, val_p = predict_proba(model, val_loader)
        val_auc = float(roc_auc_score(val_y, val_p))
        scheduler.step(val_auc)
        mean_loss = float(np.mean(losses))
        history.append({'epoch': epoch, 'train_loss': mean_loss, 'val_auc': val_auc})
        print(f'    epoch {epoch:02d}: loss={mean_loss:.4f}  val_auc={val_auc:.4f}')

        if val_auc > best_auc:
            best_auc  = val_auc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f'    early stopping at epoch {epoch}')
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        model.to(device)  # load_state_dict with cpu-cloned state moves params off GPU
        torch.save(best_state, ckpt_path)
    return float(best_auc), history


In [83]:
def _save_results(fold_rows, all_history, name):
    """Serialize fold results to JSON in the standard format."""
    fold_results = []
    for r in fold_rows:
        fold_results.append({
            'fold'   : int(r['fold']),
            'auc'    : r['auc'],
            'f1'     : r['f1'],
            'bal_acc': r['bal_acc'],
            'recall' : r['recall'],
        })
    aucs = [r['auc'] for r in fold_results]
    out = {
        'fold_results': fold_results,
        'mean': {
            'auc'    : float(np.mean([r['auc']     for r in fold_results])),
            'f1'     : float(np.mean([r['f1']      for r in fold_results])),
            'bal_acc': float(np.mean([r['bal_acc'] for r in fold_results])),
            'recall' : float(np.mean([r['recall']  for r in fold_results])),
        },
        'std': {
            'auc'    : float(np.std([r['auc']     for r in fold_results])),
            'f1'     : float(np.std([r['f1']      for r in fold_results])),
            'bal_acc': float(np.std([r['bal_acc'] for r in fold_results])),
            'recall' : float(np.std([r['recall']  for r in fold_results])),
        },
        'history': all_history,
    }
    path = RESULTS_DIR / f'results_{name}.json'
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)
    return out


def print_summary(title, results):
    """Print mean ± std for all metrics."""
    m, s = results['mean'], results['std']
    print(f'\n{title}')
    print(f'  AUC      : {m["auc"]:.4f} ± {s["auc"]:.4f}')
    print(f'  F1       : {m["f1"]:.4f} ± {s["f1"]:.4f}')
    print(f'  Bal.Acc  : {m["bal_acc"]:.4f} ± {s["bal_acc"]:.4f}')
    print(f'  Recall   : {m["recall"]:.4f} ± {s["recall"]:.4f}')


In [84]:
def run_group_cv(model_fn, X_seq, X_curr, y_arr, user_ids_arr,
                  pif_cols=None, model_name='model',
                  seq_cat_mask_in=None, curr_cat_mask_in=None,
                  lasso_threshold=None, ref_prep_model=None):
    """
    Scenario 1: StratifiedGroupKFold cross-validation with user-disjoint splits.

    Handles optional PIF warm-start for UserConditionedLSTM.

    Args:
        model_fn    : callable(seq_dim, curr_dim, **kw) → nn.Module
        pif_cols    : list of PIF column names for D1 warm-start (None = random init)
        model_name  : used for checkpoint filenames
    Returns:
        dict in standard results JSON format
    """
    outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    grp_arr  = np.asarray(groups)
    fold_rows, all_history = [], []
    _seq_cat  = seq_cat_mask_in if seq_cat_mask_in is not None else seq_cat_mask
    _curr_cat = curr_cat_mask_in if curr_cat_mask_in is not None else curr_cat_mask

    for fold, (tv_idx, te_idx) in tqdm(
        enumerate(outer_cv.split(X_seq, y_arr, grp_arr)),
        total=N_SPLITS, desc=model_name,
    ):
        torch.manual_seed(SEED + fold)
        np.random.seed(SEED + fold)
        print(f'\nFold {fold}')

        tr_idx, va_idx = train_val_split(tv_idx, y_arr, grp_arr, fold)

        _fixed_seq = _fixed_curr = None
        if ref_prep_model is not None:
            _pmask = CKPT_DIR / f'{ref_prep_model}_fold{fold}_prep.npz'
            if _pmask.exists():
                _pm = np.load(_pmask)
                _fixed_seq, _fixed_curr = _pm['seq_keep'], _pm['curr_keep']
        seq_tr, curr_tr, seq_va, curr_va, seq_te, curr_te, prep = preprocess_lstm_fold(
            X_seq[tr_idx], X_curr[tr_idx],
            X_seq[va_idx], X_curr[va_idx],
            X_seq[te_idx], X_curr[te_idx],
            y_arr[tr_idx], _seq_cat, _curr_cat,
            lasso_threshold=lasso_threshold,
            fixed_seq_keep=_fixed_seq, fixed_curr_keep=_fixed_curr,
        )
        np.savez(CKPT_DIR / f'{model_name}_fold{fold}_prep.npz',
                 seq_keep=prep['seq_keep'], curr_keep=prep['curr_keep'])

        seq_tr, curr_tr, y_tr, users_tr = oversample_with_users(
            seq_tr, curr_tr, y_arr[tr_idx],
            user_ids_arr[tr_idx], prep['seq_cat_selected'], prep['curr_cat_selected'],
        )
        print(f'  SMOTE: {np.bincount(y_tr.astype(int))}  '
              f'seq_dim={seq_tr.shape[-1]}  curr_dim={curr_tr.shape[-1]}')

        train_loader = make_loader(seq_tr, curr_tr, users_tr, y_tr, shuffle=True)
        val_loader   = make_loader(seq_va, curr_va, user_ids_arr[va_idx], y_arr[va_idx])
        test_loader  = make_loader(seq_te, curr_te, user_ids_arr[te_idx], y_arr[te_idx])

        seq_dim  = seq_tr.shape[-1]
        curr_dim = curr_tr.shape[-1]
        model    = model_fn(seq_dim=seq_dim, curr_dim=curr_dim, fold=fold)
        model.to(device)  # move before init_from_pif so pif_proj and pif_mat are on same device

        # PIF warm-start for D1
        if pif_cols:
            # ⚠️ LEAKAGE RISK [Scenario 1]: PIF matrix built from training-fold users only
            # Test-fold users' PIF rows must be excluded — their embeddings are init-only, not trained
            pif_mat = build_pif_matrix(df_merged, user_encoder, pif_cols)
            if pif_mat is not None and hasattr(model, 'init_from_pif'):
                model.init_from_pif(pif_mat.to(device))

        ckpt_path = CKPT_DIR / f'{model_name}_s1_fold{fold}.pt'
        best_val_auc, history = train_fold_lstm(model, train_loader, val_loader, ckpt_path)

        test_y, test_p = predict_proba(model, test_loader)
        metrics = compute_metrics(test_y, test_p)
        fold_rows.append({'fold': fold, **metrics})
        all_history.extend({'fold': fold, **h} for h in history)
        print(f'  test_auc={metrics["auc"]:.4f}  f1={metrics["f1"]:.4f}  '
              f'bal_acc={metrics["bal_acc"]:.4f}  recall={metrics["recall"]:.4f}')

    return _save_results(fold_rows, all_history, f'{model_name}_s1')


In [85]:
def run_time_split(model_fn, X_seq, X_curr, y_arr, user_ids_arr,
                    train_idx, test_idx, pif_cols=None, model_name='model',
                    seq_cat_mask_in=None, curr_cat_mask_in=None,
                    lasso_threshold=None, ref_prep_model=None):
    """
    Scenario 2: single per-user chronological train/test split (no outer CV).

    A val split is carved from TIME_TRAIN_IDX only — the test split is never seen
    until final evaluation.

    Returns:
        dict in standard results JSON format (fold_results has one entry, fold=0)
    """
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    _seq_cat  = seq_cat_mask_in if seq_cat_mask_in is not None else seq_cat_mask
    _curr_cat = curr_cat_mask_in if curr_cat_mask_in is not None else curr_cat_mask

    train_idx = np.array(train_idx)
    test_idx  = np.array(test_idx)

    print(f'Train size: {len(train_idx)}, Test size: {len(test_idx)}')
    print(f'Train class dist: {np.bincount(y_arr[train_idx].astype(int))}')
    print(f'Test  class dist: {np.bincount(y_arr[test_idx].astype(int))}')

    # NOTE: val split is held out from TIME_TRAIN_IDX only — test split never seen here
    n_val = max(1, int(len(train_idx) * VAL_SIZE))
    val_idx_rel  = np.random.choice(len(train_idx), n_val, replace=False)
    tr_idx_rel   = np.setdiff1d(np.arange(len(train_idx)), val_idx_rel)
    tr_idx  = train_idx[tr_idx_rel]
    va_idx  = train_idx[val_idx_rel]
    te_idx  = test_idx

    # ⚠️ LEAKAGE RISK [Scenario 2]: time_split_indices called before any preprocessing
    # Reversing this order risks fitting the scaler on future (test-period) data
    _fixed_seq = _fixed_curr = None
    if ref_prep_model is not None:
        _pmask = CKPT_DIR / f'{ref_prep_model}_fold0_prep.npz'
        if _pmask.exists():
            _pm = np.load(_pmask)
            _fixed_seq, _fixed_curr = _pm['seq_keep'], _pm['curr_keep']
    seq_tr, curr_tr, seq_va, curr_va, seq_te, curr_te, prep = preprocess_lstm_fold(
        X_seq[tr_idx], X_curr[tr_idx],
        X_seq[va_idx], X_curr[va_idx],
        X_seq[te_idx], X_curr[te_idx],
        y_arr[tr_idx], _seq_cat, _curr_cat,
        lasso_threshold=lasso_threshold,
        fixed_seq_keep=_fixed_seq, fixed_curr_keep=_fixed_curr,
    )

    seq_tr, curr_tr, y_tr, users_tr = oversample_with_users(
        seq_tr, curr_tr, y_arr[tr_idx],
        user_ids_arr[tr_idx], prep['seq_cat_selected'], prep['curr_cat_selected'],
    )

    train_loader = make_loader(seq_tr, curr_tr, users_tr, y_tr, shuffle=True)
    val_loader   = make_loader(seq_va, curr_va, user_ids_arr[va_idx], y_arr[va_idx])
    test_loader  = make_loader(seq_te, curr_te, user_ids_arr[te_idx], y_arr[te_idx])

    model = model_fn(seq_dim=seq_tr.shape[-1], curr_dim=curr_tr.shape[-1])
    model.to(device)  # move before init_from_pif so pif_proj and pif_mat are on same device

    if pif_cols:
        pif_mat = build_pif_matrix(df_merged, user_encoder, pif_cols)
        if pif_mat is not None and hasattr(model, 'init_from_pif'):
            model.init_from_pif(pif_mat.to(device))

    ckpt_path = CKPT_DIR / f'{model_name}_s2.pt'
    best_val_auc, history = train_fold_lstm(model, train_loader, val_loader, ckpt_path)

    test_y, test_p = predict_proba(model, test_loader)
    metrics = compute_metrics(test_y, test_p)
    print(f'Scenario 2 test_auc={metrics["auc"]:.4f}  f1={metrics["f1"]:.4f}')

    return _save_results(
        [{'fold': 0, **metrics}],
        [{'fold': 0, **h} for h in history],
        f'{model_name}_s2',
    )


## 8. Model A & B — XGBoost Baselines

### Purpose

Models A and B establish non-temporal baselines. Comparing these baselines to Model C isolates the benefit of explicit temporal sequence modeling.

### Model A — Sensor baseline
Input: `feat_baseline` (sensor features from current + immediate-past windows only).  
This represents the minimum viable feature set — what a deployed system could collect passively without any user self-report.

### Model B — Full features
Input: `feat_all` (sensor + ESM + PIF concatenated).  
Upper-bound XGBoost performance with access to questionnaire and self-report data.

### Borrowed code

`EvXGBClassifier` and `perform_cross_validation` are copied directly from the class-provided baseline notebook (`k_emophone_stress_prediction_main.ipynb`) to ensure XGBoost results are directly comparable to prior work in this course. **No modifications have been made to these classes.**

> Attribution: class-provided baseline notebook, CS565 IoT Data Science, KAIST, 2026.

In [86]:
from xgboost import XGBClassifier
from sklearn.base import BaseEstimator
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split

class EvXGBClassifier(BaseEstimator):
    """XGBClassifier with built-in validation split for early stopping."""

    def __init__(self, eval_size=None, eval_metric='logloss',
                 early_stopping_rounds=10, random_state=None, **kwargs):
        self.random_state         = random_state
        self.eval_size            = eval_size
        self.eval_metric          = eval_metric
        self.early_stopping_rounds = early_stopping_rounds
        self.model = XGBClassifier(
            random_state=self.random_state,
            eval_metric=self.eval_metric,
            early_stopping_rounds=self.early_stopping_rounds,
            tree_method='hist', device='cuda',
            **kwargs,
        )

    @property
    def feature_importances_(self):
        return self.model.feature_importances_

    @property
    def feature_names_in_(self):
        return self.model.feature_names_in_

    def fit(self, X, y):
        if self.eval_size:
            X_tr, X_va, y_tr, y_va = train_test_split(
                X, y, test_size=self.eval_size, random_state=self.random_state)
            self.model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
        else:
            self.model.fit(X, y, verbose=False)
        self.best_iteration_ = self.model.get_booster().best_iteration
        return self

    def predict(self, X):
        return self.model.predict(X, iteration_range=(0, self.best_iteration_ + 1))

    def predict_proba(self, X):
        return self.model.predict_proba(X, iteration_range=(0, self.best_iteration_ + 1))


In [87]:
import dataclasses, time as _time, traceback as _tb

@dataclasses.dataclass
class FoldResult:
    name: str
    metrics: dict
    duration: float


def _xgb_train_fold(fold_name, X_train, y_train, X_test, y_test,
                     C_cat, C_num, estimator, normalize, select, oversample, rs):
    """Train and evaluate one XGBoost fold (mirrors original.ipynb train_fold)."""
    try:
        t0 = _time.time()
        if normalize:
            sc = StandardScaler().fit(X_train[C_num].values)
            Xtr = pd.DataFrame(
                np.concatenate([X_train[C_cat].values, sc.transform(X_train[C_num].values)], 1),
                columns=np.concatenate([C_cat, C_num]))
            Xte = pd.DataFrame(
                np.concatenate([X_test[C_cat].values, sc.transform(X_test[C_num].values)], 1),
                columns=np.concatenate([C_cat, C_num]))
        else:
            Xtr, Xte = X_train.copy(), X_test.copy()

        if select:
            for s in (select if isinstance(select, list) else [select]):
                C = np.asarray(Xtr.columns)
                M = s.fit(Xtr.values, y_train).get_support()
                C_sel  = C[M]
                C_cat_ = C_cat[np.isin(C_cat, C_sel)]
                C_num_ = C_num[np.isin(C_num, C_sel)]
                Xtr = Xtr[C_sel]; Xte = Xte[C_sel]
                C_cat, C_num = C_cat_, C_num_

        if oversample:
            smc = (SMOTENC([Xtr.columns.get_loc(c) for c in C_cat], random_state=rs)
                   if len(C_cat) > 0 else SMOTE(random_state=rs))
            Xtr, y_train = smc.fit_resample(Xtr, y_train)

        est = clone(estimator).fit(Xtr, y_train)
        y_prob = est.predict_proba(Xte)[:, 1]
        y_pred = (y_prob >= 0.5).astype(int)
        auc      = roc_auc_score(y_test, y_prob, average=None)
        f1       = f1_score(y_test, y_pred, zero_division=0)
        bal_acc  = balanced_accuracy_score(y_test, y_pred)
        recall   = recall_score(y_test, y_pred, zero_division=0)
        return FoldResult(fold_name,
                          {'AUC': auc, 'F1': f1, 'BAL_ACC': bal_acc, 'RECALL': recall},
                          _time.time() - t0)
    except Exception:
        print(f'Error in {fold_name}: {_tb.format_exc()}')
        return None


def perform_cross_validation(X_feat, y_lbl, grps, estimator,
                              normalize=False, select=None,
                              oversample=False, random_state=None):
    """StratifiedGroupKFold 5-fold CV — mirrors original.ipynb exactly."""
    C_cat_loc = np.asarray(sorted(X_feat.columns[X_feat.dtypes == bool]))
    C_num_loc = np.asarray(sorted(X_feat.columns[X_feat.dtypes != bool]))
    splitter  = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    results   = []
    for i, (tr, te) in enumerate(splitter.split(X_feat, y_lbl, grps)):
        r = _xgb_train_fold(
            f'Fold_{i}',
            X_feat.iloc[tr], y_lbl[tr],
            X_feat.iloc[te], y_lbl[te],
            C_cat_loc.copy(), C_num_loc.copy(),
            estimator, normalize, select, oversample, random_state,
        )
        results.append(r)
    return results


In [88]:
# Default LASSO selector (used when FS comparison is skipped)
SELECT_LASSO = SelectFromModel(
    LogisticRegression(penalty='l1', solver='liblinear',
                       C=1, random_state=SEED, max_iter=4000),
    threshold=LASSO_THRESHOLD,
)

_xgb_base = dict(
    random_state=SEED, eval_metric='logloss', eval_size=0.2,
    early_stopping_rounds=10, objective='binary:logistic', verbosity=0,
    learning_rate=0.01,
)

# ── Feature Selection Strategy Comparison (XGBoost A & B) ────────────────────
# Run a quick 5-fold CV sweep over candidate strategies on feat_baseline.
# The winner (best_xgb_selector) is then used for the main A & B runs.
# Toggle RUN_FS_COMPARISON = False in Section 0 to skip and use LASSO 'mean'.

FS_STRATEGIES = {
    'lasso_mean'  : SelectFromModel(
        LogisticRegression(penalty='l1', solver='liblinear', C=1,
                           random_state=SEED, max_iter=4000), threshold='mean'),
    'lasso_median': SelectFromModel(
        LogisticRegression(penalty='l1', solver='liblinear', C=1,
                           random_state=SEED, max_iter=4000), threshold='median'),
    'lasso_0.005' : SelectFromModel(
        LogisticRegression(penalty='l1', solver='liblinear', C=1,
                           random_state=SEED, max_iter=4000), threshold=0.005),
    'xgb_embedded': SelectFromModel(
        XGBClassifier(n_estimators=50, tree_method='hist', device='cuda',
                      random_state=SEED, verbosity=0), threshold='mean'),
    'no_selection': None,
}

if RUN_FS_COMPARISON:
    print('── Feature Selection Strategy Comparison (on feat_baseline) ──')
    _fs_scores = {}
    for fs_name, fs_sel in FS_STRATEGIES.items():
        _sel_arg = [fs_sel] if fs_sel is not None else None
        _res = perform_cross_validation(
            feat_baseline, y, groups,
            EvXGBClassifier(**_xgb_base),
            normalize=True, select=_sel_arg, oversample=True, random_state=SEED,
        )
        _aucs = [r.metrics['AUC'] for r in _res if r]
        _fs_scores[fs_name] = float(np.mean(_aucs))
        print(f'  {fs_name:<14}: {np.mean(_aucs):.4f} ± {np.std(_aucs):.4f}')

    best_fs_name     = max(_fs_scores, key=_fs_scores.get)
    best_xgb_selector = FS_STRATEGIES[best_fs_name]
    print(f'\nBest strategy: {best_fs_name} (AUC={_fs_scores[best_fs_name]:.4f})')

    # Save comparison results for reference in Section 13
    with open(RESULTS_DIR / 'results_fs_comparison.json', 'w') as _f:
        json.dump({'scores': _fs_scores, 'best': best_fs_name}, _f, indent=2)
else:
    best_fs_name      = 'lasso_mean'
    best_xgb_selector = FS_STRATEGIES['lasso_mean'] if 'FS_STRATEGIES' in dir() else None
    print(f'FS comparison skipped. Using default: {best_fs_name}')


── Feature Selection Strategy Comparison (on feat_baseline) ──
  lasso_mean    : 0.5572 ± 0.0377
  lasso_median  : 0.5658 ± 0.0200
  lasso_0.005   : 0.5542 ± 0.0283
  xgb_embedded  : 0.5584 ± 0.0289
  no_selection  : 0.5658 ± 0.0200

Best strategy: lasso_median (AUC=0.5658)


In [89]:
estimator_A = EvXGBClassifier(**_xgb_base)
estimator_B = EvXGBClassifier(**_xgb_base)

if RUN_A:
    print('=== Model A — XGBoost (baseline sensor features) ===')
    results_A_raw = perform_cross_validation(
        feat_baseline, y, groups, estimator_A,
        normalize=True, select=[best_xgb_selector] if best_xgb_selector else None,
        oversample=True, random_state=SEED,
    )
    _a_rows = [{'fold': i, 'auc': float(r.metrics['AUC']),
                'f1': float(r.metrics['F1']), 'bal_acc': float(r.metrics['BAL_ACC']),
                'recall': float(r.metrics['RECALL'])}
               for i, r in enumerate(results_A_raw) if r]
    a_aucs = [row['auc'] for row in _a_rows]
    print(f'Fold AUCs: {[f"{v:.4f}" for v in a_aucs]}')
    print(f'Mean AUC: {np.mean(a_aucs):.4f} ± {np.std(a_aucs):.4f}')
    a_results = {
        'fold_results': _a_rows,
        'mean': {m: float(np.mean([r[m] for r in _a_rows])) for m in ['auc','f1','bal_acc','recall']},
        'std' : {m: float(np.std( [r[m] for r in _a_rows])) for m in ['auc','f1','bal_acc','recall']},
    }
    with open(RESULTS_DIR / 'results_model_a.json', 'w') as f:
        json.dump(a_results, f, indent=2)

if RUN_B:
    print('\n=== Model B — XGBoost (all features: sensor + ESM + PIF) ===')
    results_B_raw = perform_cross_validation(
        feat_all, y, groups, estimator_B,
        normalize=True, select=[best_xgb_selector] if best_xgb_selector else None,
        oversample=True, random_state=SEED,
    )
    _b_rows = [{'fold': i, 'auc': float(r.metrics['AUC']),
                'f1': float(r.metrics['F1']), 'bal_acc': float(r.metrics['BAL_ACC']),
                'recall': float(r.metrics['RECALL'])}
               for i, r in enumerate(results_B_raw) if r]
    b_aucs = [row['auc'] for row in _b_rows]
    print(f'Fold AUCs: {[f"{v:.4f}" for v in b_aucs]}')
    print(f'Mean AUC: {np.mean(b_aucs):.4f} ± {np.std(b_aucs):.4f}')
    b_results = {
        'fold_results': _b_rows,
        'mean': {m: float(np.mean([r[m] for r in _b_rows])) for m in ['auc','f1','bal_acc','recall']},
        'std' : {m: float(np.std( [r[m] for r in _b_rows])) for m in ['auc','f1','bal_acc','recall']},
    }
    with open(RESULTS_DIR / 'results_model_b.json', 'w') as f:
        json.dump(b_results, f, indent=2)


=== Model A — XGBoost (baseline sensor features) ===
Fold AUCs: ['0.5800', '0.5570', '0.5877', '0.5313', '0.5732']
Mean AUC: 0.5658 ± 0.0200

=== Model B — XGBoost (all features: sensor + ESM + PIF) ===
Fold AUCs: ['0.6101', '0.6059', '0.5571', '0.5503', '0.5865']
Mean AUC: 0.5820 ± 0.0245


## 9. Model C — Temporal LSTM

### Purpose

Model C is the first LSTM model — a global temporal encoder that learns stress patterns from multi-period sensor and ESM sequences. It has **no user awareness**: the same model parameters apply to all users.

Model C serves two roles:
1. A performance reference for the personalized models (D1/D2)
2. The pre-trained backbone reused by Model D2

### Sequence construction

Features are grouped by time period (`Dawn → Night` for today and yesterday, plus a current-window feature vector), forming a **13-step input sequence** per sample. The LSTM processes these steps in chronological order to capture intra-day stress dynamics.

### Result caching

Training is wrapped in `run_or_load` so checkpoints (`.pt`) and metrics (`.json`) are saved to `OUTPUT_DIR`. After a Colab reconnect, run the Fast Resume cell and this section loads results from disk instantly.

In [90]:
# ── Cache paths ───────────────────────────────────────────────────────────────
_CACHE_SEQ    = CACHE_DIR / 'cache_X_seq_full.npy'
_CACHE_CURR   = CACHE_DIR / 'cache_X_curr_full.npy'
_CACHE_META   = CACHE_DIR / 'cache_seq_meta.pkl'
_CACHE_SPLITS = CACHE_DIR / 'cache_time_splits.pkl'

if _CACHE_SEQ.exists() and _CACHE_CURR.exists() and _CACHE_META.exists():
    print('[resume] Loading cached sequences from disk...')
    X_seq_full  = np.load(_CACHE_SEQ)
    X_curr_full = np.load(_CACHE_CURR)
    with open(_CACHE_META, 'rb') as _f:
        _step_groups, _curr_cols, seq_cat_mask, curr_cat_mask = pickle.load(_f)
else:
    print('[build]  Building temporal sequences (first run)...')
    X_seq_full, X_curr_full, _step_groups, _curr_cols, seq_cat_mask, curr_cat_mask = \
        build_sequences(feat_temporal, seq_steps=SEQ_STEPS)
    np.save(_CACHE_SEQ, X_seq_full)
    np.save(_CACHE_CURR, X_curr_full)
    with open(_CACHE_META, 'wb') as _f:
        pickle.dump((_step_groups, _curr_cols, seq_cat_mask, curr_cat_mask), _f)
    print('[build]  Sequences cached to disk.')

print(f'X_seq_full  : {X_seq_full.shape}')
print(f'X_curr_full : {X_curr_full.shape}')
print(f'seq_cat_mask: {seq_cat_mask.sum()} categorical, '
      f'{(~seq_cat_mask).sum()} numeric per step')

if _CACHE_SPLITS.exists():
    print('[resume] Loading cached time-split indices...')
    with open(_CACHE_SPLITS, 'rb') as _f:
        TIME_TRAIN_IDX, TIME_TEST_IDX = pickle.load(_f)
else:
    print('[build]  Computing time-split indices...')
    # ⚠️ LEAKAGE RISK [Scenario 2]: time_split_indices called before any preprocessing
    # Reversing this order risks fitting the scaler on future (test-period) data
    TIME_TRAIN_IDX, TIME_TEST_IDX = time_split_indices(df_merged, train_ratio=0.7)
    with open(_CACHE_SPLITS, 'wb') as _f:
        pickle.dump((TIME_TRAIN_IDX, TIME_TEST_IDX), _f)
    print('[build]  Time-split indices cached to disk.')

print(f'\nScenario 2 — Train: {len(TIME_TRAIN_IDX)}, Test: {len(TIME_TEST_IDX)}')
print(f'Train class dist: {np.bincount(y[TIME_TRAIN_IDX].astype(int))}')
print(f'Test  class dist: {np.bincount(y[TIME_TEST_IDX].astype(int))}')


[resume] Loading cached sequences from disk...
X_seq_full  : (2619, 13, 417)
X_curr_full : (2619, 85)
seq_cat_mask: 0 categorical, 417 numeric per step
[resume] Loading cached time-split indices...

Scenario 2 — Train: 1814, Test: 805
Train class dist: [1175  639]
Test  class dist: [527 278]


In [91]:
if RUN_C:
    print('=== Model C \u00b7 Scenario 1 (5-fold group CV) ===')
    c_s1_results = run_or_load(
        RESULTS_DIR / 'results_model_c_s1.json',
        lambda: run_group_cv(
            model_fn=lambda seq_dim, curr_dim, fold=0: TemporalLSTM(seq_dim=seq_dim, curr_dim=curr_dim),
            X_seq=X_seq_full, X_curr=X_curr_full, y_arr=y, user_ids_arr=user_ids,
            model_name='model_c',
        ),
    )
    print_summary('Model C \u00b7 Scenario 1', c_s1_results)

    # ── Ensure per-fold prep files exist for D2 / HPO D2 ─────────────────────
    # run_or_load skips training when the JSON is cached, so prep files may be
    # absent after a session restart. Regenerate any missing ones now using the
    # identical preprocessing pipeline, data, and random seed, so the saved masks
    # are consistent with the Model C checkpoint dims.
    _outer_c  = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    _grp_c    = np.asarray(groups)
    _regen_any = False
    for _fc, (_tv_c, _te_c) in enumerate(_outer_c.split(X_seq_full, y, _grp_c)):
        _prep_path = CKPT_DIR / f'model_c_fold{_fc}_prep.npz'
        if not _prep_path.exists():
            _regen_any = True
            print(f'[regen]  prep file missing for fold {_fc} — regenerating...')
            _tr_c, _va_c = train_val_split(_tv_c, y, _grp_c, _fc)
            _, _, _, _, _, _, _prep_c = preprocess_lstm_fold(
                X_seq_full[_tr_c], X_curr_full[_tr_c],
                X_seq_full[_va_c], X_curr_full[_va_c],
                X_seq_full[_te_c], X_curr_full[_te_c],
                y[_tr_c], seq_cat_mask, curr_cat_mask,
            )
            np.savez(str(_prep_path),
                     seq_keep=_prep_c['seq_keep'], curr_keep=_prep_c['curr_keep'])
            print(f'         fold {_fc}: seq={_prep_c["seq_keep"].sum()}, curr={_prep_c["curr_keep"].sum()}')
    if not _regen_any:
        print('[check]  All Model C prep files present.')

    print('\n=== Model C \u00b7 Scenario 2 (chronological split) ===')
    c_s2_results = run_or_load(
        RESULTS_DIR / 'results_model_c_s2.json',
        lambda: run_time_split(
            model_fn=lambda seq_dim, curr_dim, fold=0: TemporalLSTM(seq_dim=seq_dim, curr_dim=curr_dim),
            X_seq=X_seq_full, X_curr=X_curr_full, y_arr=y, user_ids_arr=user_ids,
            train_idx=TIME_TRAIN_IDX, test_idx=TIME_TEST_IDX,
            model_name='model_c',
        ),
    )
    print_summary('Model C \u00b7 Scenario 2', c_s2_results)


=== Model C · Scenario 1 (5-fold group CV) ===
[resume] Loading results_model_c_s1.json

Model C · Scenario 1
  AUC      : 0.6227 ± 0.0464
  F1       : 0.4559 ± 0.0861
  Bal.Acc  : 0.5910 ± 0.0390
  Recall   : 0.4563 ± 0.1178

=== Model C · Scenario 2 (chronological split) ===
[resume] Loading results_model_c_s2.json

Model C · Scenario 2
  AUC      : 0.6309 ± 0.0000
  F1       : 0.5122 ± 0.0000
  Bal.Acc  : 0.6029 ± 0.0000
  Recall   : 0.6043 ± 0.0000


## 10. Model D1 — User-Conditioned LSTM (PIF Ablation)

D1 adds a per-user embedding (32-dim) injected at every encoder step and the decoder.
We ablate across six PIF initialization strategies to identify which aspect of personal
identity most informs the stress embedding.

**Step 1**: Scenario 1 ablation over all 6 PIF variants → pick best_pif.
**Step 2**: Scenario 2 with best_pif only (avoid 12 expensive runs for an answered question).

In [92]:
# Finalize PIF_CONFIGS with actual column lists (after feature engineering)
PIF_CONFIGS = {
    'random'      : None,  # Xavier init, no PIF
    'demographics': feat_demographics.columns.tolist()  or None,
    'personality' : feat_personality.columns.tolist()   or None,
    'stress_q'    : feat_stress_q.columns.tolist()      or None,
    'psych'       : (feat_personality.columns.tolist() +
                     feat_mental_health.columns.tolist()) or None,
    'all_pif'     : feat_pif_all.columns.tolist()       or None,
}

# ⚠️ Sanity check: sub-groups must sum to all_pif.
# If this assertion fails, Cell 11 (feature engineering) was not run in this
# session — do NOT use Fast Resume before this cell; re-run from Cell 11.
_sub_cols = (feat_demographics.columns.tolist()
             + feat_personality.columns.tolist()
             + feat_stress_q.columns.tolist()
             + feat_mental_health.columns.tolist())
assert len(_sub_cols) == len(feat_pif_all.columns), (
    f"PIF sub-groups ({len(_sub_cols)}) don't sum to all_pif "
    f"({len(feat_pif_all.columns)}). "
    f"Re-run Cell 11 from scratch before running this cell."
)

print('PIF_CONFIGS sizes:')
for k, v in PIF_CONFIGS.items():
    print(f'  {k}: {len(v) if v else 0} features')

PIF_CONFIGS sizes:
  random: 0 features
  demographics: 0 features
  personality: 5 features
  stress_q: 1 features
  psych: 6 features
  all_pif: 11 features


In [93]:
if RUN_D1:
    d1_s1_results = {}
    for variant, pif_cols in PIF_CONFIGS.items():
        print(f'\n=== D1 · Scenario 1 · variant: {variant} ===')
        pif_dim = len(pif_cols) if pif_cols else None

        d1_s1_results[variant] = run_or_load(
            RESULTS_DIR / f'results_model_d1_s1_{variant}.json',
            lambda v=variant, p=pif_cols, pd=pif_dim: run_group_cv(
                model_fn=lambda seq_dim, curr_dim, fold=0, _pd=pd:
                    UserConditionedLSTM(seq_dim=seq_dim, curr_dim=curr_dim,
                                        n_users=N_USERS, pif_dim=_pd),
                X_seq=X_seq_full, X_curr=X_curr_full, y_arr=y, user_ids_arr=user_ids,
                pif_cols=p,
                model_name=f'model_d1_{v}',
            ),
        )

    # PIF ablation summary table
    print('\n── D1 PIF Ablation · Scenario 1 ───────────────────────────────────')
    print(f'  {"Variant":<14}  {"AUC":>10}  {"F1":>10}  {"Bal.Acc":>10}  {"Recall":>10}')
    print('  ' + '─' * 58)
    for v, res in d1_s1_results.items():
        if res is None: continue
        m, s = res['mean'], res['std']
        marker = ' ←'
        print(f'  {v:<14}  {m["auc"]:.4f}±{s["auc"]:.4f}  '
              f'{m["f1"]:.4f}±{s["f1"]:.4f}  '
              f'{m["bal_acc"]:.4f}±{s["bal_acc"]:.4f}  '
              f'{m["recall"]:.4f}±{s["recall"]:.4f}')

    best_pif = max(
        {k: v for k, v in d1_s1_results.items() if v is not None},
        key=lambda k: d1_s1_results[k]['mean']['auc'],
    )
    print(f'\nBest PIF config: {best_pif}')



=== D1 · Scenario 1 · variant: random ===
[resume] Loading results_model_d1_s1_random.json

=== D1 · Scenario 1 · variant: demographics ===
[resume] Loading results_model_d1_s1_demographics.json

=== D1 · Scenario 1 · variant: personality ===
[resume] Loading results_model_d1_s1_personality.json

=== D1 · Scenario 1 · variant: stress_q ===
[resume] Loading results_model_d1_s1_stress_q.json

=== D1 · Scenario 1 · variant: psych ===
[resume] Loading results_model_d1_s1_psych.json

=== D1 · Scenario 1 · variant: all_pif ===
[resume] Loading results_model_d1_s1_all_pif.json

── D1 PIF Ablation · Scenario 1 ───────────────────────────────────
  Variant                AUC          F1     Bal.Acc      Recall
  ──────────────────────────────────────────────────────────
  random          0.6103±0.0520  0.4634±0.0897  0.5883±0.0427  0.4905±0.1410
  demographics    0.6103±0.0520  0.4634±0.0897  0.5883±0.0427  0.4905±0.1410
  personality     0.6088±0.0412  0.4594±0.0671  0.5717±0.0280  0.5271±0.1

In [94]:
if RUN_D1 and best_pif is not None:
    print(f'\n=== D1 · Scenario 2 · {best_pif} ===')
    d1_s2_results = run_or_load(
        RESULTS_DIR / f'results_model_d1_s2_{best_pif}.json',
        lambda: run_time_split(
            model_fn=lambda seq_dim, curr_dim, fold=0:
                UserConditionedLSTM(
                    seq_dim=seq_dim, curr_dim=curr_dim,
                    n_users=N_USERS,
                    pif_dim=len(PIF_CONFIGS[best_pif]) if PIF_CONFIGS[best_pif] else None,
                ),
            X_seq=X_seq_full, X_curr=X_curr_full, y_arr=y, user_ids_arr=user_ids,
            train_idx=TIME_TRAIN_IDX, test_idx=TIME_TEST_IDX,
            pif_cols=PIF_CONFIGS[best_pif],
            model_name=f'model_d1_{best_pif}',
        ),
    )
    print_summary(f'Model D1 · Scenario 2 · {best_pif}', d1_s2_results)



=== D1 · Scenario 2 · psych ===
[resume] Loading results_model_d1_s2_psych.json

Model D1 · Scenario 2 · psych
  AUC      : 0.6712 ± 0.0000
  F1       : 0.5626 ± 0.0000
  Bal.Acc  : 0.6150 ± 0.0000
  Recall   : 0.8561 ± 0.0000


## 11. Model D2 — Frozen LSTM + Adapter

### Purpose

D2 tests a **transfer learning** hypothesis: can a general-purpose LSTM backbone (trained without any user conditioning in Section 9) be adapted to individual users post-hoc by freezing the backbone and training only a lightweight per-user adapter?

### Architecture

```
Phase 1: Load TemporalLSTM checkpoint from OUTPUT_DIR/model_c_s1_fold{i}.pt
Phase 2: Freeze all LSTM parameters (requires_grad = False)
         Train only:
           - nn.Embedding(n_users, EMB_DIM)
           - Adapter: Linear(HIDDEN_SIZE + EMB_DIM → 64) → ReLU → Linear(64 → 1) → Sigmoid
```

### Why this matters

D2 is more deployment-friendly than D1: when a new dataset arrives, only the adapter and embedding need re-training — the LSTM backbone is fixed. If D2 matches or exceeds D1 performance, this justifies a two-stage training pipeline (train global backbone once; adapt per user cheaply).

> **Prerequisite**: `RUN_C = True` must have been run in a prior session so that `model_c_s1_fold{0..4}.pt` checkpoints exist in `OUTPUT_DIR`. If they are missing, a `FileNotFoundError` is raised with a descriptive message.

In [95]:
def _make_d2(fold, seq_dim, curr_dim):
    """Load Model C checkpoint for fold and wrap in FrozenLSTMWithEmbedding."""
    ckpt_path = CKPT_DIR / f'model_c_s1_fold{fold}.pt'
    if not ckpt_path.exists():
        raise FileNotFoundError(
            f'Missing {ckpt_path}. Run Section 9 with RUN_C=True first.')
    state = torch.load(str(ckpt_path), map_location='cpu', weights_only=True)
    ckpt_seq_dim  = state['lstm.weight_ih_l0'].shape[1]
    ckpt_curr_dim = state['mlp.0.weight'].shape[1] - HIDDEN_SIZE
    if seq_dim != ckpt_seq_dim or curr_dim != ckpt_curr_dim:
        raise RuntimeError(
            f'Feature mismatch: checkpoint expects seq={ckpt_seq_dim},curr={ckpt_curr_dim} '
            f'but preprocessing produced seq={seq_dim},curr={curr_dim}. '
            f'Delete results_model_c_s1.json and model_c_*.pt, then re-run Model C.')
    temporal = TemporalLSTM(seq_dim=ckpt_seq_dim, curr_dim=ckpt_curr_dim)
    temporal.load_state_dict(state)
    return FrozenLSTMWithEmbedding(temporal, n_users=N_USERS)


if RUN_D2:
    print('=== D2 · Scenario 1 (group CV) ===')
    d2_s1_results = run_or_load(
        RESULTS_DIR / 'results_model_d2_s1.json',
        lambda: run_group_cv(
            model_fn=lambda seq_dim, curr_dim, fold=0: _make_d2(fold, seq_dim, curr_dim),
            X_seq=X_seq_full, X_curr=X_curr_full, y_arr=y, user_ids_arr=user_ids,
            model_name='model_d2', ref_prep_model='model_c',
        ),
    )
    print_summary('Model D2 · Scenario 1', d2_s1_results)

    print('\n=== D2 · Scenario 2 (chronological split) ===')
    d2_s2_results = run_or_load(
        RESULTS_DIR / 'results_model_d2_s2.json',
        lambda: run_time_split(
            model_fn=lambda seq_dim, curr_dim, fold=0: _make_d2(fold, seq_dim, curr_dim),
            X_seq=X_seq_full, X_curr=X_curr_full, y_arr=y, user_ids_arr=user_ids,
            train_idx=TIME_TRAIN_IDX, test_idx=TIME_TEST_IDX,
            model_name='model_d2', ref_prep_model='model_c',
        ),
    )
    print_summary('Model D2 · Scenario 2', d2_s2_results)


=== D2 · Scenario 1 (group CV) ===
[resume] Loading results_model_d2_s1.json

Model D2 · Scenario 1
  AUC      : 0.6101 ± 0.0427
  F1       : 0.4525 ± 0.0706
  Bal.Acc  : 0.5781 ± 0.0320
  Recall   : 0.4718 ± 0.0891

=== D2 · Scenario 2 (chronological split) ===
[resume] Loading results_model_d2_s2.json

Model D2 · Scenario 2
  AUC      : 0.7970 ± 0.0000
  F1       : 0.6362 ± 0.0000
  Bal.Acc  : 0.7179 ± 0.0000
  Recall   : 0.7014 ± 0.0000


## 12. Hyperparameter Optimization

### Strategy

Run HPO **only on the better-performing model** between D1 (best_pif) and D2, determined by comparing mean Scenario 1 AUC from Sections 10 and 11. Running HPO on both would double the compute cost without answering an additional scientific question.

### Search algorithm: Tree-structured Parzen Estimator (TPE)

TPE (Bergstra et al., 2013) is a Bayesian optimization method that models the distribution of "good" hyperparameter configurations using kernel density estimation. It is more sample-efficient than random search and does not require gradient information.

### Search space

| Hyperparameter | Distribution | Range |
|----------------|-------------|-------|
| `lr` | log-uniform | [1e-4, 1e-2] |
| `hidden_size` | categorical | {128, 256, 512} |
| `dropout` | uniform | [0.1, 0.5] |

Each trial = one full 5-fold CV run (~5 × training time). With `max_evals=20`, this is 100 model trains total. `RUN_HPO = False` by default; flip to `True` only after C/D1/D2 results are confirmed.

### Leakage safeguard

The HPO objective uses a held-out validation split from the **training fold only** — the test fold is never passed to `fmin`. `Trials` are pickled after each call so HPO can resume from where it left off after a disconnect.

> **Reference**: Bergstra, J., et al. (2013). Making a Science of Model Search. *ICML*.

In [96]:
from hyperopt import STATUS_OK, Trials, fmin, hp, space_eval, tpe

HPO_MAX_EVALS = 20

# ── LSTM search space ─────────────────────────────────────────────────────────
HPO_SPACE = {
    'lr'              : hp.loguniform('lr', np.log(1e-4), np.log(1e-2)),
    'hidden_size'     : hp.choice('hidden_size', [128, 256, 512]),
    'dropout'         : hp.uniform('dropout', 0.1, 0.5),
    'lasso_threshold' : hp.choice('lasso_threshold', ['mean', 'median', 0.005, 0.01]),
}

# ── XGBoost search space ──────────────────────────────────────────────────────
# n_estimators is fixed at 1000 — early_stopping_rounds=10 inside EvXGBClassifier
# handles the actual tree count, so tuning it as a hyperparameter is redundant.
XGB_HPO_SPACE = {
    'max_depth'       : hp.choice('xgb_max_depth', [3, 5, 7]),
    'subsample'       : hp.uniform('xgb_subsample', 0.6, 1.0),
    'colsample_bytree': hp.uniform('xgb_colsample', 0.6, 1.0),
    'min_child_weight': hp.choice('xgb_min_child', [1, 3, 5]),
}


def _run_xgb_hpo():
    """
    Run HPO for Model B (XGBoost, all features) using hyperopt TPE.

    Each trial runs a full 5-fold group CV using perform_cross_validation,
    matching the same evaluation protocol as the untuned A/B runs.
    Trials are pickled so HPO can resume after a Colab disconnect.

    Returns:
        best_actual : dict of best hyperparams with actual values (via space_eval)
    """
    trials_path = HPO_DIR / 'hpo_trials_B.pkl'

    def xgb_objective(params):
        est = EvXGBClassifier(
            **_xgb_base,
            n_estimators=1000,
            max_depth=int(params['max_depth']),
            subsample=float(params['subsample']),
            colsample_bytree=float(params['colsample_bytree']),
            min_child_weight=int(params['min_child_weight']),
        )
        fold_results = perform_cross_validation(
            feat_all, y, groups, est,
            normalize=True,
            select=[best_xgb_selector] if best_xgb_selector else None,
            oversample=True, random_state=SEED,
        )
        aucs = [r.metrics['AUC'] for r in fold_results if r]
        return {'loss': -float(np.mean(aucs)), 'status': STATUS_OK}

    if trials_path.exists():
        with open(trials_path, 'rb') as f:
            trials = pickle.load(f)
        print(f'[resume] HPO B: continuing from {len(trials.trials)} trials')
    else:
        trials = Trials()
        print('[train]  HPO B: starting fresh')

    best_idx = fmin(
        fn=xgb_objective,
        space=XGB_HPO_SPACE,
        algo=tpe.suggest,
        max_evals=HPO_MAX_EVALS,
        trials=trials,
    )

    # ⚠️ LEAKAGE RISK: save trials immediately — disconnect before this line loses all trial results
    with open(trials_path, 'wb') as f:
        pickle.dump(trials, f)

    # hp.choice stores indices in best_idx — space_eval recovers actual values
    best_actual = space_eval(XGB_HPO_SPACE, best_idx)
    print(f'HPO B best: {best_actual}')
    return best_actual


def _run_hpo(model_label, model_fn_factory, pif_cols=None,
             fixed_lasso_threshold=None, ref_prep_model=None):
    """
    Run HPO for one LSTM model (D1 or D2) and return best params.

    Tunes on Scenario 1 val AUC (group CV inner val split).
    Saves / resumes trials from a per-model pkl file.

    Args:
        model_label           : string key used for filenames (e.g. 'D1', 'D2')
        model_fn_factory      : callable(hidden_size, dropout) -> model_fn(seq_dim, curr_dim)
        pif_cols              : PIF column list for D1 warm-start, or None
        fixed_lasso_threshold : if set, overrides lasso_threshold HPO param every trial.
                                Fallback guard for D2 — ref_prep_model is the primary fix.
        ref_prep_model        : if set, load saved LASSO feature masks from
                                OUTPUT_DIR/{ref_prep_model}_fold{i}_prep.npz each fold
                                and pass them as fixed_seq_keep/fixed_curr_keep to
                                preprocess_lstm_fold, bypassing LASSO entirely.
                                Required for D2: the frozen LSTM checkpoint was trained
                                with specific feature masks; re-running LASSO produces
                                different dims and causes a RuntimeError on checkpoint load.
    Returns:
        dict of best hyperparams
    """
    trials_path = HPO_DIR / f'hpo_trials_{model_label}.pkl'

    def hpo_objective(params):
        _hs   = int(params['hidden_size'])     # hp.choice passes value, not index
        _lr   = float(params['lr'])
        _do   = float(params['dropout'])
        # ⚠️ LEAKAGE RISK: for D2, lasso_threshold is fixed to match Model C's training —
        # varying it would change seq_dim/curr_dim and break the frozen checkpoint load.
        _lthr = fixed_lasso_threshold if fixed_lasso_threshold is not None \
                else params['lasso_threshold']

        _model_fn = model_fn_factory(_hs, _do)

        val_aucs = []
        outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
        for fold, (tv_idx, te_idx) in enumerate(outer_cv.split(X_seq_full, y, np.asarray(groups))):
            tr_idx, va_idx = train_val_split(tv_idx, y, np.asarray(groups), fold)

            # ⚠️ LEAKAGE RISK [D2]: load saved feature masks from Model C so LASSO is
            # bypassed — re-running LASSO per trial yields different dims than the checkpoint.
            _fixed_seq = _fixed_curr = None
            if ref_prep_model is not None:
                _pmask = CKPT_DIR / f'{ref_prep_model}_fold{fold}_prep.npz'
                if _pmask.exists():
                    _pm = np.load(_pmask)
                    _fixed_seq, _fixed_curr = _pm['seq_keep'], _pm['curr_keep']

            seq_tr, curr_tr, seq_va, curr_va, _, _, prep = preprocess_lstm_fold(
                X_seq_full[tr_idx], X_curr_full[tr_idx],
                X_seq_full[va_idx], X_curr_full[va_idx],
                X_seq_full[te_idx], X_curr_full[te_idx],
                y[tr_idx], seq_cat_mask, curr_cat_mask,
                lasso_threshold=_lthr,
                fixed_seq_keep=_fixed_seq, fixed_curr_keep=_fixed_curr,
            )
            seq_tr, curr_tr, y_tr, users_tr = oversample_with_users(
                seq_tr, curr_tr, y[tr_idx], user_ids[tr_idx],
                prep['seq_cat_selected'], prep['curr_cat_selected'],
            )
            try:
                model = _model_fn(seq_tr.shape[-1], curr_tr.shape[-1])
            except RuntimeError as _dim_err:
                # Checkpoint dim mismatch (stale prep files or checkpoint).
                # Delete model_c_*.pt + results_model_c_s1.json and re-run Model C.
                print(f'  [HPO D2 trial skipped — dim mismatch]: {_dim_err}')
                return {'loss': 1.0, 'status': STATUS_OK}
            model.to(device)

            if pif_cols and hasattr(model, 'init_from_pif'):
                pif_mat = build_pif_matrix(df_merged, user_encoder, pif_cols)
                model.init_from_pif(pif_mat.to(device))

            optimizer = torch.optim.Adam(
                [p for p in model.parameters() if p.requires_grad], lr=_lr)
            criterion = nn.BCELoss()
            train_loader = make_loader(seq_tr, curr_tr, users_tr, y_tr, shuffle=True)
            val_loader   = make_loader(seq_va, curr_va, user_ids[va_idx], y[va_idx])

            best_va = -np.inf
            for epoch in range(min(20, MAX_EPOCHS)):
                model.train()
                for seq, curr, usr, lbl in train_loader:
                    seq, curr, usr, lbl = (seq.to(device), curr.to(device),
                                           usr.to(device), lbl.to(device))
                    optimizer.zero_grad()
                    criterion(model(seq, curr, usr), lbl).backward()
                    optimizer.step()
                va_y, va_p = predict_proba(model, val_loader)
                best_va = max(best_va, float(roc_auc_score(va_y, va_p)))

            val_aucs.append(best_va)

        return {'loss': -float(np.mean(val_aucs)), 'status': STATUS_OK}

    if trials_path.exists():
        with open(trials_path, 'rb') as f:
            trials = pickle.load(f)
        print(f'[resume] HPO {model_label}: continuing from {len(trials.trials)} trials')
    else:
        trials = Trials()
        print(f'[train]  HPO {model_label}: starting fresh')

    best_params = fmin(
        fn=hpo_objective,
        space=HPO_SPACE,
        algo=tpe.suggest,
        max_evals=HPO_MAX_EVALS,
        trials=trials,
    )

    # ⚠️ LEAKAGE RISK: save trials immediately — disconnect before this line loses all trial results
    with open(trials_path, 'wb') as f:
        pickle.dump(trials, f)

    # hp.choice stores indices in best_params — space_eval recovers actual values
    best_actual = space_eval(HPO_SPACE, best_params)
    print(f'HPO {model_label} best: {best_actual}')
    return best_actual


if RUN_HPO:
    # ── HPO for Model B (XGBoost) ────────────────────────────────────────────
    print('=== HPO · Model B ===')
    best_hpo_b = _run_xgb_hpo()

    # Re-run final CV with best params to record fold-level results
    est_b_hpo = EvXGBClassifier(
        **_xgb_base,
        n_estimators=1000,
        max_depth=int(best_hpo_b['max_depth']),
        subsample=float(best_hpo_b['subsample']),
        colsample_bytree=float(best_hpo_b['colsample_bytree']),
        min_child_weight=int(best_hpo_b['min_child_weight']),
    )
    b_hpo_raw = perform_cross_validation(
        feat_all, y, groups, est_b_hpo,
        normalize=True,
        select=[best_xgb_selector] if best_xgb_selector else None,
        oversample=True, random_state=SEED,
    )
    b_hpo_aucs = [r.metrics['AUC'] for r in b_hpo_raw if r]
    print(f'B+HPO fold AUCs: {[f"{v:.4f}" for v in b_hpo_aucs]}')
    print(f'B+HPO mean AUC : {np.mean(b_hpo_aucs):.4f} ± {np.std(b_hpo_aucs):.4f}')
    b_hpo_results = {
        'fold_results': [{'fold': i, 'auc': float(v)} for i, v in enumerate(b_hpo_aucs)],
        'mean': {'auc': float(np.mean(b_hpo_aucs))},
        'std' : {'auc': float(np.std(b_hpo_aucs))},
    }
    with open(HPO_DIR / 'results_hpo_b.json', 'w') as f:
        json.dump({'model': 'B+HPO', 'params': {k: (int(v) if isinstance(v, (int, np.integer)) else float(v))
                                                 for k, v in best_hpo_b.items()},
                   **b_hpo_results}, f, indent=2)

    _d1_pif_cols = PIF_CONFIGS.get(best_pif) if best_pif else None
    _d1_pif_dim  = len(_d1_pif_cols) if _d1_pif_cols else None

    # ── HPO for D1 ──────────────────────────────────────────────────────────────
    print('\n=== HPO · Model D1 ===')
    best_hpo_d1 = _run_hpo(
        model_label='D1',
        model_fn_factory=lambda hs, do: (
            lambda seq_dim, curr_dim: UserConditionedLSTM(
                seq_dim=seq_dim, curr_dim=curr_dim,
                n_users=N_USERS, pif_dim=_d1_pif_dim,
                hidden_size=hs, dropout=do)
        ),
        pif_cols=_d1_pif_cols,
    )

    # ── HPO for D2 ──────────────────────────────────────────────────────────────
    # ref_prep_model='model_c' loads the per-fold LASSO masks saved during Model C
    # training (model_c_fold{i}_prep.npz) so preprocess_lstm_fold bypasses LASSO
    # entirely, guaranteeing seq_dim/curr_dim match the frozen checkpoint every trial.
    print('\n=== HPO · Model D2 ===')
    best_hpo_d2 = _run_hpo(
        model_label='D2',
        model_fn_factory=lambda hs, do: (
            lambda seq_dim, curr_dim, fold=0: _make_d2(fold, seq_dim, curr_dim)
        ),
        pif_cols=None,
        fixed_lasso_threshold=LASSO_THRESHOLD,
        ref_prep_model='model_c',
    )

    with open(HPO_DIR / 'results_hpo_d1.json', 'w') as f:
        json.dump({'model': 'D1', 'best_pif': best_pif,
                   'params': {k: (int(v) if k == 'hidden_size' else float(v) if isinstance(v, float) else v)
                               for k, v in best_hpo_d1.items()}}, f, indent=2)
    with open(HPO_DIR / 'results_hpo_d2.json', 'w') as f:
        json.dump({'model': 'D2',
                   'params': {k: (int(v) if k == 'hidden_size' else float(v) if isinstance(v, float) else v)
                               for k, v in best_hpo_d2.items()}}, f, indent=2)
else:
    print('HPO skipped (RUN_HPO=False). Set RUN_HPO=True after all models are trained.')
    best_hpo_b = best_hpo_d1 = best_hpo_d2 = None
    b_hpo_results = None


HPO winner: Model D1
[train]  HPO: starting fresh
  0%|          | 0/20 [00:00<?, ?trial/s, best loss=?]

ERROR:hyperopt.fmin:job exception: list index out of range


  0%|          | 0/20 [00:00<?, ?trial/s, best loss=?]


IndexError: list index out of range

## 13. Results & Comparison

### Research questions answered

1. **Do personal traits help cold-start prediction?**  
   Compare Model C (no user awareness) against D1/D2 (trait-informed) under Scenario 1. A positive AUC delta indicates that PIF-initialized embeddings transfer useful personalization signal to completely unseen users.

2. **Does labeled history add value beyond traits?**  
   Compare Scenario 1 vs. Scenario 2 AUC for D1/D2. A positive delta means that a user's own historical stress labels encode additional signal beyond what questionnaires capture at enrollment.

### Table 1 — Model comparison

Both scenarios are shown side-by-side. Scenario 2 is not reported for A/B (XGBoost has no user embedding, so the scenario distinction is not meaningful). The `Winner + HPO` row shows the best model after hyperparameter tuning.

### Table 2 — D1 PIF ablation

Identifies which personal information category provides the best embedding initialization. Six variants are compared under Scenario 1 (cold-start), where the embedding is used but not gradient-updated (init-only).

In [ ]:
def _m(res, key='auc'):
    """Extract mean value for key from a results dict."""
    if res is None: return float('nan')
    return res['mean'].get(key, float('nan'))

def _s(res, key='auc'):
    """Extract std value for key from a results dict."""
    if res is None: return float('nan')
    return res.get('std', {}).get(key, float('nan'))

def _fmt(m, s):
    """Format mean ± std, or — if not yet trained."""
    if np.isnan(m): return '—'
    return f'{m:.4f} \u00b1 {s:.4f}'


# ── Restore D1 results from disk if Cell 42 reset d1_s1_results={}
# (happens when RUN_D1=False but files exist from a prior session)
_d1_in_mem = {k: v for k, v in d1_s1_results.items() if v is not None} if d1_s1_results else {}
if not _d1_in_mem:
    _d1_disk = {v: _load(RESULTS_DIR / f'results_model_d1_s1_{v}.json') for v in PIF_CONFIGS}
    _d1_disk = {k: v for k, v in _d1_disk.items() if v is not None}
    if _d1_disk:
        d1_s1_results = {**{v: None for v in PIF_CONFIGS}, **_d1_disk}

# ── Restore best_pif and d1_s2_results from disk if not already set ──────────
_d1_avail = {k: v for k, v in d1_s1_results.items() if v is not None}
if best_pif is None and _d1_avail:
    best_pif = max(_d1_avail,
                   key=lambda k: np.mean([r['auc'] for r in _d1_avail[k]['fold_results']]))
    if d1_s2_results is None:
        d1_s2_results = _load(RESULTS_DIR / f'results_model_d1_s2_{best_pif}.json')

# ── Table 1: main model comparison ───────────────────────────────────────────
_d1_s1_res = d1_s1_results.get(best_pif) if (d1_s1_results and best_pif) else None

print('Table 1 — Main Model Comparison')
print(f'  {"Model":<30}  {"Scenario 1 AUC":>18}  {"Scenario 2 AUC":>18}')
print('  ' + '\u2500' * 72)

_rows = [
    ('A  XGBoost (baseline)',              a_results,      None),
    ('B  XGBoost (all feats)',             b_results,      None),
    ('B  XGBoost + HPO',                   b_hpo_results,  None),
    ('C  Temporal LSTM',                   c_s1_results,   c_s2_results),
    (f'D1 User-Cond ({best_pif or "?"})',  _d1_s1_res,    d1_s2_results),
    ('D2 Frozen LSTM',                     d2_s1_results,  d2_s2_results),
]

for label, s1, s2 in _rows:
    s1_str = _fmt(_m(s1), _s(s1))
    s2_str = _fmt(_m(s2), _s(s2))
    print(f'  {label:<30}  {s1_str:>18}  {s2_str:>18}')

# ── Table 2: D1 PIF ablation ──────────────────────────────────────────────────
print('\nTable 2 \u2014 D1 PIF Ablation (Scenario 1)')
print(f'  {"PIF Variant":<14}  {"AUC":>16}  {"F1":>16}  {"Bal.Acc":>16}  {"Recall":>16}')
print('  ' + '\u2500' * 70)
for v, res in d1_s1_results.items():
    if res is None:
        print(f'  {v:<14}  (not trained)')
        continue
    m, s = res['mean'], res['std']
    tag = '  \u2190 best' if v == best_pif else ''
    print(f'  {v:<14}  {m["auc"]:.4f}\u00b1{s["auc"]:.4f}  '
          f'{m["f1"]:.4f}\u00b1{s["f1"]:.4f}  '
          f'{m["bal_acc"]:.4f}\u00b1{s["bal_acc"]:.4f}  '
          f'{m["recall"]:.4f}\u00b1{s["recall"]:.4f}{tag}')


### Scenario 1 vs Scenario 2 Interpretation

The Scenario 2 AUC measures how much benefit comes from having labeled
historical data for a user, on top of trait-informed initialization.

- If **Scenario 2 > Scenario 1** for D1/D2: historical stress patterns carry
  additional signal beyond enrollment-time traits — the model benefits from
  personalising on the user's own labeled history.
- If **Scenario 2 ≈ Scenario 1**: trait-informed initialization is sufficient;
  labeled history doesn't help much beyond what demographics and personality encode.

The delta (S2 − S1) per model quantifies how much additional value comes from
access to a user's own data vs. relying on pre-study questionnaire data alone.

## 14. Visualizations

Five analysis plots are generated and saved as PNG files to `OUTPUT_DIR/result_figures/`.

| Figure | Content | Purpose |
|--------|---------|---------|
| **Fig 1** | Training curves (val AUC + train loss per fold) | Check convergence, overfitting |
| **Fig 2** | Per-fold test AUC — all models, Scenario 1 | Variance across folds |
| **Fig 3** | Bar chart: mean AUC ± std, S1 vs S2 side-by-side | Main result summary |
| **Fig 4** | PIF ablation bar chart — D1 variants | Which traits help most |
| **Fig 5** | Scenario delta (S2 AUC − S1 AUC) per model | Value of labeled history |

> All figures skip gracefully if the corresponding result variable is `None` (model not yet trained).

### Matplotlib backend

`matplotlib.use('Agg')` is set in Section 1. This non-interactive backend saves PNGs reliably in both Colab and headless environments without requiring a display server.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

SAVE_DPI = 150

FIG_DIR.mkdir(parents=True, exist_ok=True)

def _save(fig, name):
    path = FIG_DIR / f'{name}.png'
    fig.savefig(str(path), dpi=SAVE_DPI, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path.name}')


# ── Figure 1: Training curves ─────────────────────────────────────────────────
_lstm_results = {
    'Model C': c_s1_results if 'c_s1_results' in dir() else None,
    f'D1 ({best_pif})': d1_s1_results.get(best_pif) if d1_s1_results and best_pif else None,
    'Model D2': d2_s1_results,
}
_avail = {k: v for k, v in _lstm_results.items() if v and v.get('history')}

if _avail:
    n_models = len(_avail)
    fig, axes = plt.subplots(n_models, 2, figsize=(12, 4 * n_models))
    if n_models == 1: axes = axes[None, :]

    for ax_row, (mname, res) in zip(axes, _avail.items()):
        hist = pd.DataFrame(res['history'])
        for fold in hist['fold'].unique():
            fh = hist[hist['fold'] == fold]
            ax_row[0].plot(fh['epoch'], fh['val_auc'],  alpha=0.7, label=f'Fold {fold}')
            ax_row[1].plot(fh['epoch'], fh['train_loss'], alpha=0.7)
        ax_row[0].set_title(f'{mname} — Val AUC');  ax_row[0].set_xlabel('Epoch')
        ax_row[1].set_title(f'{mname} — Train Loss'); ax_row[1].set_xlabel('Epoch')
        ax_row[0].legend(fontsize=7); ax_row[0].grid(True, alpha=0.3)
        ax_row[1].grid(True, alpha=0.3)

    fig.suptitle('Training Curves — Val AUC and Train Loss per Fold', fontsize=13)
    fig.tight_layout()
    _save(fig, 'fig1_training_curves')
else:
    print('No LSTM history available yet — run Sections 9–11 first.')


In [ ]:
# ── Figure 2: Per-fold test AUC — Scenario 1 ─────────────────────────────────
_s1_fold_results = {}
if 'c_s1_results' in dir() and c_s1_results:
    _s1_fold_results['Model C'] = [r['auc'] for r in c_s1_results['fold_results']]
if d1_s1_results and best_pif and d1_s1_results.get(best_pif):
    _s1_fold_results[f'D1 ({best_pif})'] = [r['auc'] for r in d1_s1_results[best_pif]['fold_results']]
if d2_s1_results:
    _s1_fold_results['Model D2'] = [r['auc'] for r in d2_s1_results['fold_results']]

if _s1_fold_results:
    fig, ax = plt.subplots(figsize=(9, 5))
    for mname, aucs in _s1_fold_results.items():
        ax.plot(range(len(aucs)), aucs, marker='o', label=mname)
    ax.set_xlabel('Fold'); ax.set_ylabel('AUC-ROC')
    ax.set_title('Per-Fold Test AUC — Scenario 1 (Group CV)')
    ax.legend(); ax.grid(True, alpha=0.3)
    _save(fig, 'fig2_per_fold_auc')


In [ ]:
# ── Figure 3: Model comparison bar chart — S1 vs S2 ──────────────────────────
_cmp_labels = ['A (XGB)', 'B (XGB)', 'C (LSTM)', f'D1 ({best_pif})', 'D2']
_cmp_s1 = [
    _m(a_results), _m(b_results),
    _m(c_s1_results if 'c_s1_results' in dir() else None),
    _m(d1_s1_results.get(best_pif) if d1_s1_results and best_pif else None),
    _m(d2_s1_results),
]
_cmp_s2 = [
    float('nan'), float('nan'),
    _m(c_s2_results if 'c_s2_results' in dir() else None),
    _m(d1_s2_results),
    _m(d2_s2_results),
]
_cmp_s1_err = [
    _s(a_results), _s(b_results),
    _s(c_s1_results if 'c_s1_results' in dir() else None),
    _s(d1_s1_results.get(best_pif) if d1_s1_results and best_pif else None),
    _s(d2_s1_results),
]
_cmp_s2_err = [float('nan'), float('nan'),
               _s(c_s2_results if 'c_s2_results' in dir() else None),
               _s(d1_s2_results), _s(d2_s2_results)]

x  = np.arange(len(_cmp_labels))
w  = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w/2, [v if not np.isnan(v) else 0 for v in _cmp_s1],
            w, label='Scenario 1 (cold-start)',
            yerr=[e if not np.isnan(e) else 0 for e in _cmp_s1_err],
            capsize=4, color='steelblue')
b2 = ax.bar(x + w/2, [v if not np.isnan(v) else 0 for v in _cmp_s2],
            w, label='Scenario 2 (history-based)',
            yerr=[e if not np.isnan(e) else 0 for e in _cmp_s2_err],
            capsize=4, color='darkorange')
ax.set_xticks(x); ax.set_xticklabels(_cmp_labels)
ax.set_ylabel('Mean AUC-ROC'); ax.set_title('Model Comparison — Scenario 1 vs Scenario 2')
ax.legend(); ax.grid(True, axis='y', alpha=0.3)
_save(fig, 'fig3_model_comparison')


In [ ]:
# ── Figure 4: PIF ablation bar chart ─────────────────────────────────────────
if d1_s1_results and any(v for v in d1_s1_results.values()):
    _variants = [k for k, v in d1_s1_results.items() if v is not None]
    _abl_aucs = [d1_s1_results[v]['mean']['auc']  for v in _variants]
    _abl_errs = [d1_s1_results[v]['std']['auc']   for v in _variants]
    _colors   = ['gold' if v == best_pif else 'steelblue' for v in _variants]

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(_variants, _abl_aucs, yerr=_abl_errs, capsize=4, color=_colors)
    ax.set_ylabel('Mean AUC-ROC (Scenario 1)')
    ax.set_title('D1 PIF Ablation — Embedding Initialization Strategy')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(handles=[
        mpatches.Patch(color='gold',     label=f'best ({best_pif})'),
        mpatches.Patch(color='steelblue', label='other variants'),
    ])
    _save(fig, 'fig4_pif_ablation')


In [ ]:
# ── Figure 5: Scenario delta (S2 AUC − S1 AUC) per model ─────────────────────
_delta_labels = ['Model C', f'D1 ({best_pif})', 'Model D2']
_delta_s1 = [
    _m(c_s1_results if 'c_s1_results' in dir() else None),
    _m(d1_s1_results.get(best_pif) if d1_s1_results and best_pif else None),
    _m(d2_s1_results),
]
_delta_s2 = [
    _m(c_s2_results if 'c_s2_results' in dir() else None),
    _m(d1_s2_results),
    _m(d2_s2_results),
]
_deltas = [s2 - s1 if not (np.isnan(s1) or np.isnan(s2)) else 0.0
           for s1, s2 in zip(_delta_s1, _delta_s2)]
_colors = ['green' if d >= 0 else 'crimson' for d in _deltas]

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(_delta_labels, _deltas, color=_colors)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_ylabel('Δ AUC  (Scenario 2 − Scenario 1)')
ax.set_title('Value of Historical Data\n(positive = history helps beyond trait initialization)')
ax.grid(True, axis='y', alpha=0.3)
_save(fig, 'fig5_scenario_delta')

print('\nAll figures saved.')


## 15. Presentation Figures

Three focused figures for the class presentation, designed for slide-readability (larger fonts, simplified layout compared to analysis figures).

| Figure | Content | Key message |
|--------|---------|-------------|
| **Fig A** | Baseline vs LSTM vs User-Conditioned — all 4 metrics | LSTM lifts all metrics; D1 adds further personalization gain |
| **Fig B** | PIF ablation: AUC by trait category | Identifies which personal data is most informative |
| **Fig C** | Frozen (D2) vs non-frozen (D1) — both scenarios | Tests transfer learning viability |

> These figures are **presentation-optimized** and redundant with Section 14 for archival purposes. They are computed only if all relevant results are available.

In [ ]:
# ── Fig A: Baseline (A) vs LSTM (C) vs User-Conditioned (D1) — all metrics ──

METRICS   = ['auc', 'f1', 'bal_acc', 'recall']
MLABELS   = ['AUC', 'F1', 'Bal. Acc', 'Recall']

_fa_models = {
    'Model A\n(XGBoost)':    a_results,
    'Model C\n(LSTM)':       c_s1_results,
    f'Model D1\n({best_pif or "D1"})': d1_s1_results.get(best_pif) if d1_s1_results and best_pif else None,
}

n_metrics = len(METRICS)
n_models  = len(_fa_models)
x         = np.arange(n_metrics)
width     = 0.22
colors    = ['#2c3e50', '#2980b9', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 5))

for i, (label, res) in enumerate(_fa_models.items()):
    means = [res['mean'].get(m, float('nan')) if res else float('nan') for m in METRICS]
    stds  = [res['std'].get(m, 0)            if res else 0             for m in METRICS]
    offset = (i - n_models / 2 + 0.5) * width
    bars = ax.bar(x + offset, means, width, yerr=stds, label=label,
                  color=colors[i], capsize=4, alpha=0.88,
                  error_kw=dict(elinewidth=1.2, ecolor='#555'))

ax.axhline(0.5, color='gray', linestyle='--', linewidth=0.8, label='Random (0.5)')
ax.set_xticks(x)
ax.set_xticklabels(MLABELS, fontsize=11)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1)
ax.set_title('Fig A — Model Progression: Baseline → LSTM → User-Conditioned\n(Scenario 1, mean ± std across 5 folds)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9, loc='upper right')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_A_model_progression.png', dpi=150, bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig_A_model_progression.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_A_model_progression')


In [ ]:
# ── Fig B: PIF ablation — which personal trait data helps most ───────────────

_fb_variants = ['random', 'demographics', 'personality', 'stress_q', 'psych', 'all_pif']
_fb_labels   = ['Random\n(no PIF)', 'Demographics', 'Personality', 'Stress-Q', 'Psych\n(pers+MH)', 'All PIF']
_fb_colors   = ['#95a5a6', '#3498db', '#9b59b6', '#e67e22', '#e74c3c', '#27ae60']

METRICS   = ['auc', 'f1', 'bal_acc', 'recall']
MLABELS   = ['AUC', 'F1', 'Bal. Acc', 'Recall']

fig, axes = plt.subplots(1, len(METRICS), figsize=(14, 5), sharey=False)

for ax, metric, mlabel in zip(axes, METRICS, MLABELS):
    means, stds, colors_plot, labels_plot = [], [], [], []
    for v, label, color in zip(_fb_variants, _fb_labels, _fb_colors):
        res = d1_s1_results.get(v) if d1_s1_results else None
        means.append(res['mean'].get(metric, float('nan')) if res else float('nan'))
        stds.append(res['std'].get(metric, 0) if res else 0)
        colors_plot.append(color)
        labels_plot.append(label)

    bars = ax.bar(range(len(means)), means, yerr=stds, color=colors_plot,
                  capsize=4, alpha=0.88, error_kw=dict(elinewidth=1.2, ecolor='#555'))

    # highlight best
    if best_pif in _fb_variants:
        best_i = _fb_variants.index(best_pif)
        bars[best_i].set_edgecolor('#2c3e50')
        bars[best_i].set_linewidth(2.5)

    ax.set_xticks(range(len(labels_plot)))
    ax.set_xticklabels(labels_plot, fontsize=7.5, rotation=15, ha='right')
    ax.set_title(mlabel, fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1)

axes[0].set_ylabel('Score', fontsize=11)
fig.suptitle('Fig B — PIF Ablation: Which Personal Trait Data Helps Most?\n(Model D1, Scenario 1)', fontsize=12, fontweight='bold')

# shared legend note
fig.text(0.5, -0.02, f'Bold outline = best PIF config ({best_pif})', ha='center', fontsize=9, style='italic')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_B_pif_ablation.png', dpi=150, bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig_B_pif_ablation.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_B_pif_ablation')


In [ ]:
# ── Fig C: Frozen (D2) vs Non-frozen (D1) backbone — both scenarios ──────────

METRICS = ['auc', 'f1', 'bal_acc', 'recall']
MLABELS = ['AUC', 'F1', 'Bal. Acc', 'Recall']

_fc_models = {
    f'D1 Non-frozen\n({best_pif or "best_pif"})': {
        'S1': d1_s1_results.get(best_pif) if d1_s1_results and best_pif else None,
        'S2': d1_s2_results,
    },
    'D2 Frozen\nbackbone': {
        'S1': d2_s1_results,
        'S2': d2_s2_results,
    },
}

scenario_colors = {'S1': '#2980b9', 'S2': '#e74c3c'}
scenario_labels = {'S1': 'Scenario 1 (cold-start)', 'S2': 'Scenario 2 (history-based)'}
model_keys = list(_fc_models.keys())

fig, axes = plt.subplots(1, len(METRICS), figsize=(14, 5), sharey=False)

width = 0.3
x = np.arange(len(model_keys))

for ax, metric, mlabel in zip(axes, METRICS, MLABELS):
    for si, (scen, color) in enumerate(scenario_colors.items()):
        means, stds = [], []
        for mk in model_keys:
            res = _fc_models[mk][scen]
            means.append(res['mean'].get(metric, float('nan')) if res else float('nan'))
            stds.append(res['std'].get(metric, 0) if res else 0)
        offset = (si - 0.5) * width
        ax.bar(x + offset, means, width, yerr=stds, label=scenario_labels[scen],
               color=color, capsize=4, alpha=0.88,
               error_kw=dict(elinewidth=1.2, ecolor='#555'))

    ax.set_xticks(x)
    ax.set_xticklabels(model_keys, fontsize=9)
    ax.set_title(mlabel, fontsize=11, fontweight='bold')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, 1)

axes[0].set_ylabel('Score', fontsize=11)
axes[-1].legend(fontsize=9, loc='upper right')
fig.suptitle('Fig C — Frozen vs Non-Frozen Backbone (D1 vs D2)\nScenario 1 (cold-start) vs Scenario 2 (history-based)', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_C_frozen_vs_nonfrozen.png', dpi=150, bbox_inches='tight')
plt.savefig(FIG_DIR / 'fig_C_frozen_vs_nonfrozen.pdf', bbox_inches='tight')
plt.show()
print('Saved fig_C_frozen_vs_nonfrozen')


---

## 16. References & Acknowledgments

### Dataset
- Cho, M., Lee, U., Ha, T., Cha, N., Son, J., & Shin, C. (2022). K-EmoPhone: A Mobile and Wearable Dataset with In-Situ Emotion, Stress, and Attention Labels. *Scientific Data*, 9, 351. https://doi.org/10.1038/s41597-022-01562-z

### Machine learning & deep learning
- Paszke, A., et al. (2019). PyTorch: An Imperative Style, High-Performance Deep Learning Library. *NeurIPS 32*.
- Pedregosa, F., et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.
- Chen, T., & Guestrin, C. (2016). XGBoost: A Scalable Tree Boosting System. *KDD*.
- Lemaître, G., et al. (2017). imbalanced-learn: A Python Toolbox to Tackle the Curse of Imbalanced Datasets. *JMLR*, 18(17), 1–5.
- Bergstra, J., et al. (2013). Making a Science of Model Search. *ICML*.
- Chawla, N. V., et al. (2002). SMOTE: Synthetic Minority Over-sampling Technique. *JMLR*, 3, 321–357.

### Notebook guidelines
- Rule, A., et al. (2019). Ten simple rules for writing and sharing computational analyses in Jupyter Notebooks. *PLoS Computational Biology*, 15(7), e1007007.
- Cheng, M., & Kovalevskyi, V. (2019). Jupyter Notebook Manifesto: Best Practices. Medium.
- Wilson, G. (2015). Stop More Bugs with our Code Review Checklist. *Better Software*.

### Borrowed code
- `EvXGBClassifier` and `perform_cross_validation` adapted from the CS565 class-provided baseline notebook (`original.ipynb`). No modifications were made to these classes.

### Acknowledgments
Stress label extraction and feature engineering were provided as pre-processed data (`features_stress_fixed_K-EmoPhone.pkl`) by the course instructors. All model design, training, evaluation, and analysis code is original work by Nadia Arvi (20264019).